In [50]:
import pandas as pd
import datetime
import pymysql
import pandas.io.sql as psql
from datetime import datetime as dt
import numpy as np
import pandas.tseries.offsets as offsets
import os
from decimal import Decimal
import calendar
import utils
from utils import *
#importlib.reload(utils)
print(os.getcwd())

import python_ss as ps

C:\Users\suehara\Desktop\お転機BOX\ぱいそん練習\yojitu


In [22]:
#!/usr/bin/env python
# coding: utf-8

import os
import ast
import json
# importでエラーが出てしまった場合は、コマンドプロンプトにて「pip install ”必要なモジュール”」でインストールしていただく必要がございます。
# 例. pip install db_dtypes
import pandas as pd
import db_dtypes
from google.cloud import bigquery
from google.oauth2 import service_account
from google.cloud import secretmanager

def access_secret_version(project_id, secret_id, version_id='latest'):
    client = secretmanager.SecretManagerServiceClient()

    name = f"projects/{project_id}/secrets/{secret_id}/versions/{version_id}"
    response = client.access_secret_version(request={"name": name})
    payload = response.payload.data.decode("UTF-8")
    return ast.literal_eval(payload)

# 上記関数を実行するコードが記載されています。こちらもそのままお使いください。
credentials = service_account.Credentials.from_service_account_info(
  access_secret_version('temp-for-sandbox', 'TEMP_CREDENTIAL_KEY'),
  scopes=["https://www.googleapis.com/auth/cloud-platform"],
)


C:\Users\suehara\Anaconda3\lib\site-packages\google\auth\_default.py:78: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [23]:
pd.options.display.max_rows = 10000
pd.options.display.max_columns = 200


In [24]:
#ロンザンのマスタデータのパスの設定
path1 = r"\\172.16.0.232\CoffeeCrazy\経営ソリューション事業部\□シニアスカウト事業部□\01 全体進捗\02 行動カレンダー\pythonデータ"

#マスタデータを読み込み
# master1= 日付・月・カレンダー週・Qデータ(2019/10/1	23-10月	9月5W(23日～1日)	23-1Q)
master1 = pd.read_excel(path1 + "\※最新※ロンザン社長資料マスタデータ.xlsx",sheet_name='マスタ',usecols=[0,1,2,3])

# master2= 在籍Q・人マスタ・略・user_id・所属フラグ・ロンザン所属フラグ(23-2Q	五十嵐奏子	五十嵐　igarashi ミドル	0)
master2 = pd.read_excel(path1 + "\※最新※ロンザン社長資料マスタデータ.xlsx",sheet_name='マスタ',usecols=[6,7,8,9,10,11])

# master3= 人マスタ・略・チーム・レイヤー(大仲研司	大仲	1課	部責)
# master3 = pd.read_excel(path1 + "\※最新※ロンザン社長資料マスタデータ.xlsx",sheet_name='マスタ',usecols=[14,15,16,17])

# master4= ヨミ表選択・丸め(人事部	人事部紹介)
master4 = pd.read_excel(path1 + "\※最新※ロンザン社長資料マスタデータ.xlsx",sheet_name='マスタ',usecols=[20,21])

# master5= 計上Q・修正後ポイント・Q計上時ポイント・掛け率(19-3Q	5,712	7,297	78%)
master5 = pd.read_excel(path1 + "\※最新※ロンザン社長資料マスタデータ.xlsx",sheet_name='マスタ',usecols=[23,24,25,26])

In [25]:
#master3 = master3.rename(columns={"人マスタ.1": "人マスタ","sei_plus.1":"sei_plus"}) #カラム名変更
#master3 = master3.dropna(subset=['人マスタ', 'sei_plus'])

master2 = master2.dropna(subset=['人マスタ', 'sei_plus'])

master4 = master4.dropna(subset=['ヨミ表選択'])

master5 = master5.rename(columns={"計上Q.1": "計上Q","掛け率.1":"掛け率"}) #カラム名変更
master5 = master5.dropna(subset=['計上Q'])

In [26]:
master5.head(50)

,計上Q,修正後ポイント,Q計上時ポイント,掛け率
0,19-3Q,5712.159302,7296.691750,0.669330
1,19-4Q,7912.021373,10210.079728,0.662559
2,20-1Q,6230.477480,7747.157985,0.687615
3,20-2Q,8306.772656,10212.205474,0.695471
4,20-3Q,9093.505730,11825.852067,0.657453
5,20-4Q,11875.329470,14196.374044,0.715211
6,21-1Q,10264.123115,11670.000000,0.751999
7,21-2Q,10638.149000,NaN,0.855000
8,21-3Q,NaN,NaN,0.855000
9,21-4Q,NaN,NaN,0.855000


In [27]:
#日付のマスタデータのパスの設定
path2 = r"\\172.16.0.232\CoffeeCrazy\総合市場開発部\50　個人フォルダ\40　【大阪】\塩澤\マスタ"
Q_master = pd.read_excel(path2 + "\Qマスタ.xlsx",usecols=[0,1,2,3,5,8,11,12,13])
Q_master["日付"] = pd.to_datetime(Q_master["日付"]) #日付データを変換
Q_master.columns

Index(['日付', 'Q', '月', 'Q同営', 'Q同旬', '営業日比較', '同営業日比較', '同旬月日比較', '期間対象外'], dtype='object')

In [28]:
Q_master

,日付,Q,月,Q同営,Q同旬,営業日比較,同営業日比較,同旬月日比較,期間対象外
0,2009-02-02,12-2Q,2009-02-01,12-2Q同営業日,12-2Q同旬月日,30,同営業日,同旬月日,対象
1,2009-02-03,12-2Q,2009-02-01,12-2Q同営業日,12-2Q同旬月日,30,同営業日,同旬月日,対象
2,2009-02-04,12-2Q,2009-02-01,12-2Q同営業日,12-2Q同旬月日,30,同営業日,同旬月日,対象
3,2009-02-05,12-2Q,2009-02-01,12-2Q同営業日,12-2Q同旬月日,30,同営業日,同旬月日,対象
4,2009-02-06,12-2Q,2009-02-01,12-2Q同営業日,12-2Q同旬月日,30,同営業日,同旬月日,対象
5,2009-02-07,12-2Q,2009-02-01,12-2Q同営業日,12-2Q同旬月日,30,同営業日,同旬月日,対象
6,2009-02-08,12-2Q,2009-02-01,12-2Q同営業日,12-2Q同旬月日,30,同営業日,同旬月日,対象
7,2009-02-09,12-2Q,2009-02-01,12-2Q同営業日,12-2Q同旬月日,30,同営業日,同旬月日,対象
8,2009-02-10,12-2Q,2009-02-01,12-2Q同営業日,12-2Q同旬月日,30,同営業日,同旬月日,対象
9,2009-02-11,12-2Q,2009-02-01,12-2Q同営業日,12-2Q同旬月日,30,同営業日,同旬月日,対象


In [29]:
#社員データ抽出
sql="""
select
user_id ,
sei_plus,
concat(sei,mei) as seimei
FROM `temp-380708.live_company.syain`
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
syain_data = client.query(sql).result().to_dataframe()
syain_data.head()

,user_id,sei_plus,seimei
0,y-iizuka,None,飯塚裕
1,ichikawa,None,市川貴教
2,inohana,None,猪鼻千賀
3,go,None,呉宛庭
4,karasaki,None,唐崎美香


In [30]:
#業務委託の方など、live_company_syain に載ってない人分はこちらにてデータを追加
d={'user_id': ['yuko-kyotani','aya-takeshita'],
  'sei_plus': ['京谷悠子','竹下綾'],
  'seimei':['京谷悠子','竹下綾']}
ronzan_syain_data = pd.DataFrame(d) 

syain_data = pd.concat([ronzan_syain_data,syain_data])
syain_data.head()

,user_id,sei_plus,seimei
0,yuko-kyotani,京谷悠子,京谷悠子
1,aya-takeshita,竹下綾,竹下綾
0,y-iizuka,None,飯塚裕
1,ichikawa,None,市川貴教
2,inohana,None,猪鼻千賀


In [31]:
conn3 = pymysql.connect(
                    host="192.168.5.124",
                    user="eigyou_kikaku",
                    password="As6hV2K!k",
                    db="eigyou_kikaku",
                    port=3306,
                    charset='utf8mb4',
                    cursorclass=pymysql.cursors.DictCursor)


In [32]:
# 今Q取得
Q = [i['Q'] for i in get_data("select `Q` from eigyoubi_master where `日付` = curdate() - interval 7 day", conn3)][0]

# 営業日情報取得
first_date = [i['日付'] for i in get_data("select `日付` from eigyoubi_master where Q = '{}' order by `日付` asc".format(Q), conn3)][0].strftime('%Y-%m-%d')
end_date = [i['日付'] for i in get_data("select `日付` from eigyoubi_master where Q = '{}' order by `日付` desc".format(Q), conn3)][0].strftime('%Y-%m-%d')

In [33]:
print(first_date)
print(end_date)
print(Q)

2023-12-28
2024-03-27
27-2Q


# 営業AP

In [34]:
##本交渉データ

sql = """
SELECT 
  hon.id as honkosho_id,
  hon.anken_id as anken_id,
  vhon.kohosha_id as kohosha_id,
  kgy.id as kigyo_id,
  kgy.name,
  consts.name as ap_source,
  format_date('%Y/%m/%d',hon.kosho_setteibi) as hon_setteibi,
  format_date('%Y/%m/%d',hon.kosho_yoteibi) as hon_yoteibi,
  format_date('%Y/%m/%d',hon.kosho_jisshibi) as hon_jisshibi,
  format_date('%Y/%m/%d',gep.seiyaku_date) as seiyakubi,
  case when syi1.sei_plus is null then hon.kohosha_tanto
  else syi1.sei_plus end as kohosha_tanto,
  syi2.sei_plus as kigyo_tanto,
  hon.kosho_seq,
  vhon.saikosho_flag
FROM `temp-380708.live_rhs.honkoshos` hon
LEFT JOIN `temp-for-sandbox.ronzanmi__mart.vw_honkosho_saikoshos` vhon on hon.id = vhon.Honkosho_id
LEFT JOIN `temp-380708.live_rhs.ankens` an on vhon.anken_id = an.id
LEFT JOIN `temp-380708.live_rhs.kigyos` kgy on an.kigyo_id = kgy.id
LEFT JOIN `temp-380708.live_rhs.kohoshas` khs on vhon.kohosha_id = khs.id
LEFT JOIN `temp-380708.live_company.syain` syi1 on hon.kohosha_tanto = syi1.user_id
LEFT JOIN `temp-380708.live_company.syain` syi2 on an.kigyo_tanto = syi2.user_id
LEFT JOIN `temp-380708.live_rhs.geppou_naitei_temp` gep on khs.sugarid = gep.candidate_no
LEFT JOIN (SELECT 
              code,
              name
             FROM `temp-380708.live_rhs.sys_consts`
             where group_code = 19) consts on khs.ap_source = consts.code
WHERE hon.kosho_setteibi >= date '2019-09-30'
ORDER BY khs.id
;
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
hon_data = client.query(sql).result().to_dataframe()

hon_data["hon_setteibi"] = pd.to_datetime(hon_data["hon_setteibi"]) #日付データを変換
hon_data["hon_yoteibi"] = pd.to_datetime(hon_data["hon_yoteibi"]) #日付データを変換
hon_data["hon_jisshibi"] = pd.to_datetime(hon_data["hon_jisshibi"]) #日付データを変換
hon_data["seiyakubi"] = pd.to_datetime(hon_data["seiyakubi"]) #日付データを変換

In [35]:
##営業データ

sql = """
  SELECT 
    ap.id as ap_id,
    ap.kigyo_id,
    ful.fulfills_date as eigyo_jisshibi,
  FROM `temp-380708.live_rhs.sales_appoints` ap
  JOIN `temp-380708.live_rhs.sales_appoint_fulfills` ful on ap.id = ful.id
  WHERE ful.fulfills_date is not null
  
;
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
eigyo_data = client.query(sql).result().to_dataframe()

eigyo_data["eigyo_jisshibi"] = pd.to_datetime(eigyo_data["eigyo_jisshibi"]) #日付データを変換

In [36]:
# データのマージ
merged_data = pd.merge(hon_data, eigyo_data, on='kigyo_id')

# setteibi以前のjisshibiデータをフィルタリング
filtered_data = merged_data[merged_data['eigyo_jisshibi'] < merged_data['hon_setteibi']]

# jisshibiでソートし、最後のデータを取得
filtered_data.sort_values(by='eigyo_jisshibi', inplace=True)
final_data = filtered_data.drop_duplicates(subset='kigyo_id', keep='last')

C:\Users\suehara\AppData\Local\Temp\ipykernel_1608\1903966547.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data.sort_values(by='eigyo_jisshibi', inplace=True)


In [37]:
final_data = final_data[final_data['hon_setteibi'] >= "2022/12/28"]

In [38]:
final_data = final_data.sort_values(by='hon_setteibi')

In [39]:

final_data.replace([np.inf, -np.inf], np.nan, inplace=True)
final_data.fillna('', inplace=True)

final_data = final_data.values.tolist()

In [58]:
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly',
          'https://www.googleapis.com/auth/spreadsheets']
json_path = r"C:\Users\suehara\Desktop\お転機BOX\ぱいそん練習\python_ss\credentials.json"
service = ps.get_auth(SCOPES,json_path)
SPREADSHEET_ID = '1-yKaCAU0AdEea4qBZg6I9Dll0pIPbkg7s9zdteHSLxU'
Sheet_NAME = 'data!A'
Sheet_row = "2"
RANGE_NAME = Sheet_NAME+Sheet_row
ps.update_ss(SPREADSHEET_ID,RANGE_NAME,final_data,service)

AttributeError: module 'python_ss' has no attribute 'get_auth'

In [ ]:
final_data

In [ ]:
#営業AP数のデータを出す
sql = """
SELECT
id,
kigyo_id,
appoint_get_syain,
appoint_visit_syain,
appoint_get_date,
visit_times as kaisu,
company_name,
appoint_source,
appoint_visit_plan_date
FROM
`temp-380708.live_rhs.sales_appoints`

where appoint_get_date >= '2017-04-01'


;
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
sales_ap_data = client.query(sql).result().to_dataframe()

sales_ap_data["appoint_get_date"] = pd.to_datetime(sales_ap_data["appoint_get_date"]) #日付データを変換
sales_ap_data["appoint_visit_plan_date"] = pd.to_datetime(sales_ap_data["appoint_visit_plan_date"]) #日付データを変換


In [ ]:
#企業初期商談設定週（カレンダー用）　追加
sales_ap_data = sales_ap_data.rename(columns={"appoint_get_date":"日付"}) #カラム名変換
sales_ap_data = pd.merge(sales_ap_data,master1,on = ("日付"),how = "left") #対面商談設定日関連の日付データを紐付け
sales_ap_data = sales_ap_data.rename(columns={"週":"対面商談設定週","日付":"appoint_get_date","appoint_get_syain":"user_id","計上Q":"Q"})
sales_ap_data = pd.merge(sales_ap_data,syain_data,on = "user_id",how="left") #名前が変換される
sales_ap_data = sales_ap_data[['id','kigyo_id','appoint_get_date','user_id',"sei_plus",'対面商談設定週','kaisu']].copy()
sales_ap_data = sales_ap_data.rename(columns={"appoint_get_date":"日付","user_id":"担当者","対面商談設定週":"ｶﾚﾝﾀﾞｰ週"})
sales_ap_data["kaisu"] =sales_ap_data["kaisu"].fillna(0)
sales_ap_data["value"] = sales_ap_data.apply(lambda x : 1 if x["kaisu"] == 1 else 0 ,axis = 1)
sales_ap_data['type'] = 'eigyou_ap'
sales_ap_data['組手'] = ''
sales_ap_data['sai_flg'] = ''
sales_ap_data['APソース丸め'] = ''


#Q情報　追加
sales_ap = pd.merge(sales_ap_data,Q_master,on = ("日付"),how = "left")


In [ ]:
df = sales_ap.copy()


In [ ]:
#訪問担当がロンザン所属外メンバーかどうか確認
df_tantosha = df.rename(columns={"担当者": "user_id"}).copy() #カラム名変更

In [ ]:
df_tantosha = pd.merge(df_tantosha,master2,on = ("sei_plus","Q","user_id"),how = "left") #名前が変換される（変換されない人は、そのQにロンザンじゃなかった）


In [ ]:
df_tantosha["ロンザン所属フラグ"] = df_tantosha["ロンザン所属フラグ"].fillna("0") #AP担当が空欄（ロンザン所属以外）のデータを埋める


In [ ]:
df_tantosha = df_tantosha.rename(columns={"ロンザン所属フラグ":"企_ロンザン所属フラグ","user_id":"kigyo_tanto","sei_plus":"企_略氏名"})


In [ ]:
df_tantosha = df_tantosha[["id","企_ロンザン所属フラグ","kumite_moto","企_略氏名"]]#必要項目のみにする


In [ ]:
#候補者担当がロンザン所属外メンバーかどうか確認
df_kohosha = df.rename(columns={"kohosha_tanto": "user_id"}).copy() #カラム名変更
df_kohosha = pd.merge(df_kohosha,syain_data,on = "user_id",how="left").drop("seimei",axis=1) #名前が変換される
df_kohosha = pd.merge(df_kohosha,master2,on = ("sei_plus","Q","user_id"),how = "left") #名前が変換される（変換されない人は、そのQにロンザンじゃなかった）
df_kohosha["ロンザン所属フラグ"] = df_kohosha["ロンザン所属フラグ"].fillna("0") #AP担当が空欄（ロンザン所属以外）のデータを埋める
df_kohosha = df_kohosha.rename(columns={"ロンザン所属フラグ":"候_ロンザン所属フラグ","user_id":"kohosha_tanto","sei_plus":"候_略氏名"})
df_kohosha = df_kohosha[["id","候_ロンザン所属フラグ","kumite_moto","候_略氏名"]]#必要項目のみにする


#企業担当がロンザン所属外メンバーかどうか確認
df_kigyo = df.rename(columns={"kigyo_tanto": "user_id"}).copy() #カラム名変更
df_kigyo = pd.merge(df_kigyo,syain_data,on = "user_id",how="left").drop("seimei",axis=1) #名前が変換される
df_kigyo = pd.merge(df_kigyo,master2,on = ("sei_plus","Q","user_id"),how = "left") #名前が変換される（変換されない人は、そのQにロンザンじゃなかった）
df_kigyo["ロンザン所属フラグ"] = df_kigyo["ロンザン所属フラグ"].fillna("0") #AP担当が空欄（ロンザン所属以外）のデータを埋める
df_kigyo = df_kigyo.rename(columns={"ロンザン所属フラグ":"企_ロンザン所属フラグ","user_id":"kigyo_tanto","sei_plus":"企_略氏名"})
df_kigyo = df_kigyo[["id","企_ロンザン所属フラグ","企_略氏名"]]#必要項目のみにする

df2 = pd.merge(df_kohosha,df_kigyo,on = "id",how="left") #名前が変換される

In [ ]:
df_tantosha

In [ ]:
master1

In [ ]:
sales_ap

In [ ]:
#営業数のデータを抽出

sql = """
SELECT
id,
kigyo_id,
company_name,
appoint_source,
appoint_visit_syain,
fulfills_date,
visit_times as kaisu
FROM
`temp-380708.live_rhs.sales_appoint_fulfills`

where fulfills_date >= '2017-04-01'

;

"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
sales_apful_data = client.query(sql).result().to_dataframe()

sales_apful_data["fulfills_date"] = pd.to_datetime(sales_apful_data["fulfills_date"]) #日付データを変換

In [ ]:
#企業営業実施週（カレンダー用）　追加
sales_apful_data = sales_apful_data.rename(columns={"fulfills_date":"日付"}) #カラム名変換
sales_apful_data = pd.merge(sales_apful_data,master1,on = ("日付"),how = "left") #対面商談設定日関連の日付データを紐付け
sales_apful_data = sales_apful_data.rename(columns={"週":"アポ実施週","日付":"fulfills_date","appoint_visit_syain":"user_id"})
sales_apful_data = pd.merge(sales_apful_data,syain_data,on = "user_id",how="left") #名前が変換される

sales_apful_data = sales_apful_data[['id','fulfills_date','user_id',"sei_plus",'アポ実施週','kaisu']].copy()
sales_apful_data = sales_apful_data.rename(columns={"fulfills_date":"日付","user_id":"担当者","アポ実施週":"ｶﾚﾝﾀﾞｰ週"})

sales_apful_data["kaisu"] =sales_apful_data["kaisu"].fillna(0)

sales_apful_data["value"] = sales_apful_data.apply(lambda x : 1 if x["kaisu"] == 1 else 0 ,axis = 1)
sales_apful_data['type'] = 'eigyou'
sales_apful_data['組手'] = ''
sales_apful_data['sai_flg'] = ''
sales_apful_data['APソース丸め'] = ''
sales_apful_data['saikosho_apsource'] = ''

#Q情報　追加
sales_apful = pd.merge(sales_apful_data,Q_master,on = ("日付"),how = "left")

sales_apful.to_excel('sales_apful.xlsx',sheet_name='new_sheet_name')

In [ ]:
sales_apful

# 初期交渉

In [ ]:
#初期交渉のDBから基本データを抽出（初期交渉データ）
#AP獲得者がいる場合は、AP担当はAP獲得者。
#AP獲得者が空欄でAPソース：パートナー紹介の場合はAP担当は紹介受領者。
#AP獲得者が空欄でAPソース：人事部、転機は面談担当者。
#AP獲得者が空欄でAPソースがパートナー紹介、人事部、転機の場合もAP担当は面談担当者。
#且つAP獲得者がロンザン所属でない場合（所属フラグ空欄）は面談担当者にする。

#sql = """
#SELECT
#a.id,
#kosho_yoteibi,
#kosho_setteibi,
#mendan_tanto,
#kosho_jisshibi,
#a.kohosha_id,
#b.seimei as kohosha_seimei,
#kohosha_sql.name as kohosha_rank,
#kosho_seq as kaisu,
#kohosha_ap_sql.name as kohosha_apsource,
#-- a.partner_id as "パートナーid",
#-- a.jinjibu_id as "人事部id",
#shokai_sql.juryosha,
#a.ap_kakutoku as shokikosho_apkakutokusha_moto,
#case
#    when a.ap_kakutoku is null and kohosha_ap_sql.name = "パートナー紹介" and shokai_sql.juryosha is not null then shokai_sql.juryosha
#    when a.ap_kakutoku is null and kohosha_ap_sql.name = "パートナー紹介" and shokai_sql.juryosha is null then a.mendan_tanto
#    when a.ap_kakutoku is null and kohosha_ap_sql.name != "パートナー紹介" then a.mendan_tanto
#    else a.ap_kakutoku end as shokikosho_apkakutokusha,
#    
#case when a.partner_id is not null then a.partner_id
#    when a.jinjibu_id is not null then a.jinjibu_id
#    else null end as juryo_id
#from
#_live_rhs__shokikoshos as a#

#left join
#(select id,seimei,kohosha_rank from _live_rhs__kohoshas) as b ON a.kohosha_id = b.id
#left join
#(select group_code,code,name from _live_rhs__sys_consts where group_code = "5200") as kohosha_sql ON b.kohosha_rank = kohosha_sql.code
#left join
#(select group_code,code,name from _live_rhs__sys_consts where group_code = "19") as kohosha_ap_sql ON a.ap_source = kohosha_ap_sql.code
#left join
#(select kohosha_id,juryosha from _live_rhs__shokaijuryos) as shokai_sql ON a.kohosha_id = shokai_sql.kohosha_id

#"""
#shokikosho_data = pd.DataFrame(get_data(sql, conn3))

#実施日の古い順に並び替え
#shokikosho_data = shokikosho_data.sort_values(by="kosho_jisshibi")

#候補者APソースが空欄の行を削除
#shokikosho_data = shokikosho_data.dropna(subset=["kohosha_apsource"])
#重複データを削除
#shokikosho_data = shokikosho_data.sort_values(['id','kosho_jisshibi']) 
#shokikosho_data = shokikosho_data.drop_duplicates(subset='id')

#shokikosho_data["kosho_yoteibi"] = pd.to_datetime(shokikosho_data["kosho_yoteibi"]) #日付データを変換
#shokikosho_data["kosho_setteibi"] = pd.to_datetime(shokikosho_data["kosho_setteibi"]) #日付データを変換
#shokikosho_data["kosho_jisshibi"] = pd.to_datetime(shokikosho_data["kosho_jisshibi"]) #日付データを変換

#shokikosho_data.to_excel('初期交渉元データ.xlsx',sheet_name='new_sheet_name')

In [ ]:
#再交渉フラグと再交渉アポソース
#sql = """
#SELECT
#id,
#kohosha_id,
#kohosha_ap_sql.name as kohosha_apsource,
#saikosho_kaisu,
#saikosho_seq,
#case when saikosho_kaisu >= 1 and saikosho_seq = 1 then "再交渉"
#else "-" end as sai_flg

#from
#_live_rhs__shokikoshos as a
#left join
#(select group_code,code,name from _live_rhs__sys_consts where group_code = "19") as kohosha_ap_sql ON a.ap_source = kohosha_ap_sql.code
#"""
#saikosho_data = pd.DataFrame(get_data(sql, conn3))

#再交渉アポソースを設定する
#saikosho_data["saikosho_apsource"] = saikosho_data.apply(lambda x : "再交渉" if x["sai_flg"] == "再交渉" else x["kohosha_apsource"] ,axis = 1)
#saikosho_data = saikosho_data[["id","saikosho_kaisu","saikosho_seq","sai_flg","saikosho_apsource"]]

In [ ]:
#初期交渉のDBから基本データを抽出（初期交渉データ）
#AP獲得者がいる場合は、AP担当はAP獲得者。
#AP獲得者が空欄でAPソース：パートナー紹介の場合はAP担当は紹介受領者。
#AP獲得者が空欄でAPソース：人事部、転機は面談担当者。
#AP獲得者が空欄でAPソースがパートナー紹介、人事部、転機の場合もAP担当は面談担当者。
#且つAP獲得者がロンザン所属でない場合（所属フラグ空欄）は面談担当者にする。

sql = """
SELECT
    a.id,
    kosho_yoteibi,
    kosho_setteibi,
    mendan_tanto,
    kosho_jisshibi,
    a.kohosha_id,
    b.seimei as kohosha_seimei,
    kohosha_sql.name as kohosha_rank,
    kosho_seq as kaisu,
    kohosha_ap_sql.name as kohosha_apsource,
-- a.partner_id as "パートナーid",
-- a.jinjibu_id as "人事部id",
    shokai_sql.juryosha,
    a.ap_kakutoku as shokikosho_apkakutokusha_moto,
case
    when a.ap_kakutoku is null and kohosha_ap_sql.name = "パートナー紹介" and shokai_sql.juryosha is not null then shokai_sql.juryosha
    when a.ap_kakutoku is null and kohosha_ap_sql.name = "パートナー紹介" and shokai_sql.juryosha is null then a.mendan_tanto
    when a.ap_kakutoku is null and kohosha_ap_sql.name != "パートナー紹介" then a.mendan_tanto
    else a.ap_kakutoku end as shokikosho_apkakutokusha,
    
case when a.partner_id is not null then a.partner_id
    when a.jinjibu_id is not null then a.jinjibu_id
    else null end as juryo_id
from `temp-380708.live_rhs.shokikoshos` as a
left join (select id,seimei,kohosha_rank from `temp-380708.live_rhs.kohoshas`) as b ON a.kohosha_id = b.id
left join (select group_code,code,name from `temp-380708.live_rhs.sys_consts` where group_code = 5200) as kohosha_sql ON b.kohosha_rank = kohosha_sql.code
left join (select group_code,code,name from `temp-380708.live_rhs.sys_consts` where group_code = 19) as kohosha_ap_sql ON a.ap_source = kohosha_ap_sql.code
left join (select kohosha_id,juryosha from `temp-380708.live_rhs.shokaijuryos`) as shokai_sql ON a.kohosha_id = shokai_sql.kohosha_id
where kosho_setteibi >= '2017-04-01'

"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
shokikosho_data = client.query(sql).result().to_dataframe()

In [ ]:
#実施日の古い順に並び替え
shokikosho_data = shokikosho_data.sort_values(by="kosho_jisshibi")

In [ ]:


#候補者APソースが空欄の行を削除
shokikosho_data = shokikosho_data.dropna(subset=["kohosha_apsource"])
#重複データを削除
shokikosho_data = shokikosho_data.sort_values(['id','kosho_jisshibi']) 
shokikosho_data = shokikosho_data.drop_duplicates(subset='id')

shokikosho_data["kosho_yoteibi"] = pd.to_datetime(shokikosho_data["kosho_yoteibi"]) #日付データを変換
shokikosho_data["kosho_setteibi"] = pd.to_datetime(shokikosho_data["kosho_setteibi"]) #日付データを変換
shokikosho_data["kosho_jisshibi"] = pd.to_datetime(shokikosho_data["kosho_jisshibi"]) #日付データを変換

#shokikosho_data.to_excel('初期交渉元データ.xlsx',sheet_name='new_sheet_name')

In [ ]:
shokikosho_data

In [ ]:
#再交渉フラグと再交渉アポソース
sql = """
SELECT
id,
kohosha_id,
kohosha_ap_sql.name as kohosha_apsource,
saikosho_kaisu,
saikosho_seq,
case when saikosho_kaisu >= 1 and saikosho_seq = 1 then "再交渉"
else "-" end as sai_flg

from
`temp-380708.live_rhs.shokikoshos` as a
left join
(select group_code,code,name from `temp-380708.live_rhs.sys_consts` where group_code = 19) as kohosha_ap_sql ON a.ap_source = kohosha_ap_sql.code
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
saikosho_data = client.query(sql).result().to_dataframe()

#再交渉アポソースを設定する
saikosho_data["saikosho_apsource"] = saikosho_data.apply(lambda x : "再交渉" if x["sai_flg"] == "再交渉" else x["kohosha_apsource"] ,axis = 1)
saikosho_data = saikosho_data[["id","saikosho_kaisu","saikosho_seq","sai_flg","saikosho_apsource"]]

In [ ]:
#初期交渉データと再交渉フラグのデータを紐付け
shokikosho_data = pd.merge(shokikosho_data,saikosho_data,on = "id",how="left") #再交渉フラグ紐付け

AP獲得者がロンザン外メンバ－時には、AP獲得者を面談担当者に置き換える

In [ ]:
df = shokikosho_data.copy()

#kosho_setteibi　をマスタデータに紐づけ
df = df.rename(columns={"kosho_setteibi":"日付"}) #カラム名変換
df = pd.merge(df,master1,on = ("日付"),how = "left") #kosho_setteibi関連の日付データを紐付け
df = df.drop(["月"], axis=1).rename(columns={"計上Q":"Q"})

#AP獲得者がロンザン所属外メンバーだった場合、AP獲得者を面談担当者に丸める（0初期交渉AP、1初期交渉実施とならないため）
df = df.rename(columns={"shokikosho_apkakutokusha": "user_id"}) #カラム名変更
df = pd.merge(df,syain_data,on = "user_id",how="left") #名前が変換される
df = pd.merge(df,master2,on = ("sei_plus","Q","user_id"),how = "left") #名前が変換される（変換されない人は、そのQにロンザンじゃなかった）

#AP担当がロンザン所属以外は面談担当者を設定する
df["ロンザン所属フラグ"]=df["ロンザン所属フラグ"].fillna("0") #AP担当が空欄（ロンザン所属以外）のデータを埋める
df["AP担当"] = df.apply(lambda x : x["user_id"] if x["ロンザン所属フラグ"] == '1' else x["mendan_tanto"] ,axis = 1) #AP担当がロンザン所属以外は面談担当者を設定する

#必要なカラム名だけに絞る（初期交渉時面談担当がロンザン所属メンバーだったか
df = df[["id","AP担当","sei_plus"]]
#df = df[["kohosha_id","AP担当","user_id","面談担当者","Q","ロンザン所属フラグ"]]
#AP獲得者データを紐付け
shokikosho_data = pd.merge(shokikosho_data,df,on = "id",how="left") 
shokikosho_data = shokikosho_data.drop(["shokikosho_apkakutokusha"], axis=1)

In [ ]:
shokikosho_data

候補者がロンザン候補者か否かを初期交渉時の候補者担当の所属で判断する


In [ ]:
#初期交渉データを交渉回数1回だけのデータに絞る
#db = shokikosho_data.query('交渉回数 == 1 or 交渉回数 == "1"').copy()
#初期交渉実施日をQマスタと紐付け
#db = db.rename(columns={"初期交渉実施日": "日付"})
#db = pd.merge(db,Q_master,on = ("日付"),how = "left")
#db = db.rename(columns={"日付":"初期交渉実施日"})
#初期交渉実施Qが空欄のデータに"-"を入れる
#db["Q"] = db["Q"].fillna("-")
#初期交渉時データに、ロンザン社員情報を紐づける
#db = db.rename(columns={"面談担当者": "user_id"}) #カラム名変更
#db = pd.merge(db,syain_data,on = "user_id",how="left") #名前が変換される
#db = pd.merge(db,master2,on = ("略氏名","Q"),how = "left") #名前が変換される（変換されない人は、そのQにロンザンじゃなかった）
#面談担当が空欄のデータに"-"を入れる
#db["ロンザン所属フラグ"]=db["ロンザン所属フラグ"].fillna("0")
#必要なカラム名だけに絞る（初期交渉時面談担当がロンザン所属メンバーだったか
#db = db[["候補者id","ロンザン所属フラグ"]]

### 初期交渉データを整える

In [ ]:
#候補者APソース　追加
shokikosho_data = shokikosho_data.rename(columns={"kohosha_apsource":"ヨミ表選択"}) #カラム名変換
shokikosho_data = pd.merge(shokikosho_data,master4,on = ("ヨミ表選択"),how = "left") #候補者APソースを紐付け
shokikosho_data = shokikosho_data.rename(columns={"丸め":"APソース丸め"})
#初期交渉設定週（カレンダー用）　追加
shokikosho_data = shokikosho_data.rename(columns={"kosho_setteibi":"日付"}) #カラム名変換
shokikosho_data = pd.merge(shokikosho_data,master1,on = ("日付"),how = "left") #kosho_setteibi関連の日付データを紐付け
shokikosho_data = shokikosho_data.rename(columns={"週":"初期交渉設定週","日付":"kosho_setteibi"})
#初期交渉実施週（カレンダー用）　追加
shokikosho_data = shokikosho_data.rename(columns={"kosho_jisshibi":"日付"}) #カラム名変換
shokikosho_data = pd.merge(shokikosho_data,master1,on = ("日付"),how = "left") #kosho_setteibi関連の日付データを紐付け
shokikosho_data = shokikosho_data.rename(columns={"週":"初期交渉実施週","日付":"kosho_jisshibi"})

In [ ]:
#初期交渉設定だけのテーブル
shokikosho_settei = shokikosho_data[['id','kosho_setteibi','AP担当','sei_plus','APソース丸め','sai_flg','kaisu','初期交渉設定週','saikosho_apsource']].copy()
shokikosho_settei = shokikosho_settei.rename(columns={"kosho_setteibi":"日付","AP担当":"担当者","初期交渉設定週":"ｶﾚﾝﾀﾞｰ週"})
shokikosho_settei["value"] = shokikosho_settei.apply(lambda x : 1 if x["kaisu"] == 1 else 0 ,axis = 1)
shokikosho_settei['type'] = 'shoki_ap'
shokikosho_settei['組手'] = ''

#初期交渉実施だけのテーブル(交渉実施日がない案件は未実施のため行削除)
shokikosho_jisshi = shokikosho_data.copy().drop(['sei_plus'],axis=1)
shokikosho_jisshi = shokikosho_jisshi.rename(columns={"mendan_tanto":"user_id"})
shokikosho_jisshi = pd.merge(shokikosho_jisshi,syain_data,on = "user_id",how="left") #名前が変換される
shokikosho_jisshi.head()

shokikosho_jisshi = shokikosho_jisshi[['id','kosho_jisshibi','user_id','sei_plus','APソース丸め','sai_flg','kaisu','初期交渉実施週','saikosho_apsource']]
shokikosho_jisshi = shokikosho_jisshi.rename(columns={"kosho_jisshibi":"日付","user_id":"担当者","初期交渉実施週":"ｶﾚﾝﾀﾞｰ週"}).dropna(subset=['日付'])
shokikosho_jisshi["value"] = shokikosho_jisshi.apply(lambda x : 1 if x["kaisu"] == 1 else 0 ,axis = 1)
shokikosho_jisshi['type'] = 'shoki_jissi'
shokikosho_jisshi['組手'] = ''



In [ ]:
#初期交渉設定と実施をそれぞれ同じ形で整える
shokikosho = pd.concat([shokikosho_settei,shokikosho_jisshi],ignore_index=True)
#Q情報　追加
shokikosho = pd.merge(shokikosho,Q_master,on = ("日付"),how = "left")

shokikosho.to_excel('shokikosho.xlsx',sheet_name='new_sheet_name')

# 本交渉設定

In [ ]:
#sql = """
#select
#a.id,
#a.anken_id,
#a.kosho_setteibi,
#b.kohosha_id,
#c.seimei as kohosha_seimei,
#a.kohosha_tanto,
#b.kigyo_tanto,
#kohosha_ap_sql.name as kohosha_apsource,
#kohosha_sql.name as kohosha_rank,
#d.name as company_name,
#a.kosho_seq as kaisu,
#a.kosho_seq_extra as absolute_1,
#a.kosho_jisshibi,
#a.mendan_tanto,
#honkosho_kumite_sql.name as kumite_moto
#from
#_live_rhs__honkoshos as a
#
#left join
#(select id,kigyo_id,kohosha_id,kigyo_tanto,kumite from _live_rhs__ankens) as b on a.anken_id = b.id
#left join
#(select id,seimei,ap_source,kohosha_rank from _live_rhs__kohoshas) as c ON b.kohosha_id = c.id
#left join
#(select id,name from _live_rhs__kigyos) as d on b.kigyo_id = d.id
#left join
#(select group_code,code,name from _live_rhs__sys_consts where group_code = "5200") as kohosha_sql ON c.kohosha_rank = kohosha_sql.code
#left join
#(select group_code,code,name from _live_rhs__sys_consts where group_code = "19") as kohosha_ap_sql ON c.ap_source = kohosha_ap_sql.code
#left join
#(select group_code,code,name from _live_rhs__sys_consts where group_code = "4500") as honkosho_kumite_sql ON b.kumite = honkosho_kumite_sql.code
#;
#"""
#honkosho_data = pd.DataFrame(get_data(sql, conn3))

#候補者APソースが空欄の行を削除
#honkosho_data = honkosho_data.dropna(subset=["kohosha_apsource"])

#honkosho_data["kosho_setteibi"] = pd.to_datetime(honkosho_data["kosho_setteibi"]) #日付データを変換
#honkosho_data["kosho_jisshibi"] = pd.to_datetime(honkosho_data["kosho_jisshibi"]) #日付データを変換

#候補者担当が"usuda","ikke","akamatsu"のデータを除外
#honkosho_data = honkosho_data[~honkosho_data["kohosha_tanto"].isin(["usuda","ikke","akamatsu"])]

In [ ]:
#再交渉フラグのデータを読み込む
#sql = """
#select
#Honkosho_id as id,
#saikosho_flag,
#case when saikosho_flag = 1 then "再交渉"
#else "-" end as sai_flg
#from
#_live_rhs__vw_honkosho_saikoshos
#"""
#saikosho_data = pd.DataFrame(get_data(sql, conn3))

In [ ]:
sql = """
select
a.id,
a.anken_id,
a.kosho_setteibi,
b.kohosha_id,
c.seimei as kohosha_seimei,
a.kohosha_tanto,
b.kigyo_tanto,
kohosha_ap_sql.name as kohosha_apsource,
kohosha_sql.name as kohosha_rank,
d.name as company_name,
a.kosho_seq as kaisu,
a.kosho_seq_extra as absolute_1,
a.kosho_jisshibi,
a.mendan_tanto,
honkosho_kumite_sql.name as kumite_moto
from
`temp-380708.live_rhs.honkoshos` as a

left join
(select id,kigyo_id,kohosha_id,kigyo_tanto,kumite from `temp-380708.live_rhs.ankens`) as b on a.anken_id = b.id
left join
(select id,seimei,ap_source,kohosha_rank from `temp-380708.live_rhs.kohoshas`) as c ON b.kohosha_id = c.id
left join
(select id,name from `temp-380708.live_rhs.kigyos`) as d on b.kigyo_id = d.id
left join
(select group_code,code,name from `temp-380708.live_rhs.sys_consts` where group_code = 5200) as kohosha_sql ON c.kohosha_rank = kohosha_sql.code
left join
(select group_code,code,name from `temp-380708.live_rhs.sys_consts` where group_code = 19) as kohosha_ap_sql ON c.ap_source = kohosha_ap_sql.code
left join
(select group_code,code,name from `temp-380708.live_rhs.sys_consts` where group_code = 4500) as honkosho_kumite_sql ON b.kumite = honkosho_kumite_sql.code

where kosho_setteibi >= '2017-04-01'

;
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
honkosho_data = client.query(sql).result().to_dataframe()

#候補者APソースが空欄の行を削除
honkosho_data = honkosho_data.dropna(subset=["kohosha_apsource"])

honkosho_data["kosho_setteibi"] = pd.to_datetime(honkosho_data["kosho_setteibi"]) #日付データを変換
honkosho_data["kosho_jisshibi"] = pd.to_datetime(honkosho_data["kosho_jisshibi"]) #日付データを変換

#候補者担当が"usuda","ikke","akamatsu"のデータを除外
honkosho_data = honkosho_data[~honkosho_data["kohosha_tanto"].isin(["usuda","ikke","akamatsu"])]

In [ ]:
#再交渉フラグのデータを読み込む
sql = """
select
Honkosho_id as id,
saikosho_flag,
case when saikosho_flag = 1 then "再交渉"
else "-" end as sai_flg
from
`temp-for-sandbox.ronzanmi__mart.vw_honkosho_saikoshos`
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
saikosho_data = client.query(sql).result().to_dataframe()

In [ ]:
#本交渉データと再交渉フラグのデータを紐付ける
honkosho_data = pd.merge(honkosho_data,saikosho_data,on = "id",how="left")
#再交渉アポソースを設定する
honkosho_data["saikosho_apsource"] = honkosho_data.apply(lambda x : "再交渉" if x["sai_flg"] == "再交渉" else x["kohosha_apsource"] ,axis = 1)
#重複データを削除
honkosho_data = honkosho_data.drop_duplicates(subset='id')

In [ ]:
honkosho_data

本交渉データに　社数別・候補者別・組手情報追加

In [ ]:
#========================================
#組手用
#========================================
df = honkosho_data.copy()
#本交渉設定日　をマスタデータに紐づけ
df = df.rename(columns={"kosho_setteibi":"日付"}) #カラム名変換
df = pd.merge(df,master1,on = ("日付"),how = "left") #kosho_setteibi関連の日付データを紐付け
df = df.drop(["月"], axis=1).rename(columns={"計上Q":"Q"})

#候補者担当がロンザン所属外メンバーかどうか確認
df_kohosha = df.rename(columns={"kohosha_tanto": "user_id"}).copy() #カラム名変更
df_kohosha = pd.merge(df_kohosha,syain_data,on = "user_id",how="left").drop("seimei",axis=1) #名前が変換される
df_kohosha = pd.merge(df_kohosha,master2,on = ("sei_plus","Q","user_id"),how = "left") #名前が変換される（変換されない人は、そのQにロンザンじゃなかった）
df_kohosha["ロンザン所属フラグ"] = df_kohosha["ロンザン所属フラグ"].fillna("0") #AP担当が空欄（ロンザン所属以外）のデータを埋める
df_kohosha = df_kohosha.rename(columns={"ロンザン所属フラグ":"候_ロンザン所属フラグ","user_id":"kohosha_tanto","sei_plus":"候_略氏名"})
df_kohosha = df_kohosha[["id","候_ロンザン所属フラグ","kumite_moto","候_略氏名"]]#必要項目のみにする

#企業担当がロンザン所属外メンバーかどうか確認
df_kigyo = df.rename(columns={"kigyo_tanto": "user_id"}).copy() #カラム名変更
df_kigyo = pd.merge(df_kigyo,syain_data,on = "user_id",how="left").drop("seimei",axis=1) #名前が変換される
df_kigyo = pd.merge(df_kigyo,master2,on = ("sei_plus","Q","user_id"),how = "left") #名前が変換される（変換されない人は、そのQにロンザンじゃなかった）
df_kigyo["ロンザン所属フラグ"] = df_kigyo["ロンザン所属フラグ"].fillna("0") #AP担当が空欄（ロンザン所属以外）のデータを埋める
df_kigyo = df_kigyo.rename(columns={"ロンザン所属フラグ":"企_ロンザン所属フラグ","user_id":"kigyo_tanto","sei_plus":"企_略氏名"})
df_kigyo = df_kigyo[["id","企_ロンザン所属フラグ","企_略氏名"]]#必要項目のみにする

df2 = pd.merge(df_kohosha,df_kigyo,on = "id",how="left") #名前が変換される

#企業担当 も 候：ロンザン×企：ロンザン=両手、 残りのケースは片手
df2["候_ロンザン所属フラグ"] = df2["候_ロンザン所属フラグ"].astype(np.int64)
df2["企_ロンザン所属フラグ"] = df2["企_ロンザン所属フラグ"].astype(np.int64)
df2["両手フラグ"] = df2["候_ロンザン所属フラグ"] * df2["企_ロンザン所属フラグ"]
df2["両手フラグ"] = df2["両手フラグ"].apply(lambda x : "両手" if x == 1 else "片手") #候補者担当、企業担当どちらもロンザンの場合は両手、それ以外は片手にする

#本交渉データの「組手」情報があればそちらを採用、本交渉データ上の「組手」がnullのときはロンザン担当者かどうかで判断した「両手フラグ」に置き換える
df2["組手"] = df2.apply(lambda x : x["kumite_moto"] if x["kumite_moto"] is not None else x["両手フラグ"] ,axis = 1)
df2 = df2.drop("kumite_moto",axis=1)
#組手情報を紐付け
honkosho_data = pd.merge(honkosho_data,df2,on = "id",how="left") 
#df2.to_excel('df2.xlsx',sheet_name='new_sheet_name')
del df2

#========================================
#候補者別用
#========================================
df3 = df[["Q","kohosha_id","id","kaisu"]].copy()
df3 = df3.loc[df3["kaisu"] == 1]

#候補者(人)別カウント用フラグ
df3["kohosha_id"] = df3["kohosha_id"].astype(str)
df3["候補者重複用"] = df3["kohosha_id"] + "★" + df3["Q"]
#重複データを表示
df3 = df3.sort_values(['候補者重複用','id'])
#重複営業にカウントしたいので、重複用&本交渉idカラムにてまずは並び替え,
df3['候補者重複'] = df3['候補者重複用'].groupby((df3['候補者重複用'] != df3['候補者重複用'].shift()).cumsum()).cumcount() + 1
df3 = df3[["id","候補者重複用","候補者重複"]]
#候補者カウント情報を紐付け
honkosho_data = pd.merge(honkosho_data,df3,on = "id",how="left")
del df3
#========================================
#社数別用
#========================================
df4 = df[["Q","kohosha_id","id","kaisu","company_name"]].copy()
df4 = df4.loc[df4["kaisu"] == 1]

#社数別カウント用フラグ
df4["社数カウント用"] = df4["Q"] + "★" + df4["company_name"]
#重複データを表示
df4 = df4.sort_values(['社数カウント用','id'])   
#重複営業にカウントしたいので、重複用&本交渉idカラムにてまずは並び替え
df4['社数カウント'] = df4['社数カウント用'].groupby((df4['社数カウント用'] != df4['社数カウント用'].shift()).cumsum()).cumcount() + 1
df4 = df4[['id','社数カウント用','社数カウント']]
#社数カウント情報を紐付け
honkosho_data = pd.merge(honkosho_data,df4,on = "id",how="left") 
del df4

### 本交渉データを整える

In [ ]:
#候補者APソース　追加
honkosho_data = honkosho_data.rename(columns={"kohosha_apsource":"ヨミ表選択"}) #カラム名変換
honkosho_data = pd.merge(honkosho_data,master4,on = ("ヨミ表選択"),how = "left") #候補者APソースを紐付け
honkosho_data = honkosho_data.rename(columns={"丸め":"APソース丸め"})
#本交渉設定週（カレンダー用）　追加
honkosho_data = honkosho_data.rename(columns={"kosho_setteibi":"日付"}) #カラム名変換
honkosho_data = pd.merge(honkosho_data,master1,on = ("日付"),how = "left") #本交渉設定日関連の日付データを紐付け
honkosho_data = honkosho_data.rename(columns={"週":"本交渉設定週","日付":"kosho_setteibi"})
#本交渉実施週（カレンダー用）　追加
honkosho_data = honkosho_data.rename(columns={"kosho_jisshibi":"日付"}) #カラム名変換
honkosho_data = pd.merge(honkosho_data,master1,on = ("日付"),how = "left") #本交渉設定日関連の日付データを紐付け
honkosho_data = honkosho_data.rename(columns={"週":"本交渉実施週","日付":"kosho_jisshibi"})

In [ ]:
#候補者担当側
#本交渉設定だけのテーブル
honkosho_settei = honkosho_data[['id','kosho_setteibi','kohosha_tanto','候_略氏名','APソース丸め','sai_flg','kaisu','組手','本交渉設定週','saikosho_apsource']].copy()
honkosho_settei = honkosho_settei.rename(columns={"kosho_setteibi":"日付","kohosha_tanto":"担当者","本交渉設定週":"ｶﾚﾝﾀﾞｰ週",'候_略氏名':'sei_plus'})
honkosho_settei["value"] = honkosho_settei.apply(lambda x : 1 if x["kaisu"] == 1 else 0 ,axis = 1)
honkosho_settei['type'] = 'settei'

honkosho_settei_kohosha = honkosho_data[['id','kosho_setteibi','kohosha_tanto','候_略氏名','APソース丸め','sai_flg','kaisu','組手','本交渉設定週','saikosho_apsource','候補者重複']].copy()
honkosho_settei_kohosha = honkosho_settei_kohosha.loc[honkosho_settei_kohosha["候補者重複"] == 1]
honkosho_settei_kohosha = honkosho_settei_kohosha.rename(columns={"kosho_setteibi":"日付","kohosha_tanto":"担当者","本交渉設定週":"ｶﾚﾝﾀﾞｰ週",'候_略氏名':'sei_plus'}).drop(["候補者重複"], axis=1)
honkosho_settei_kohosha["value"] = honkosho_settei_kohosha.apply(lambda x : 1 if x["kaisu"] == 1 else 0 ,axis = 1)
honkosho_settei_kohosha['type'] = 'settei_kohosha'

honkosho_settei_company = honkosho_data[['id','kosho_setteibi','kohosha_tanto','候_略氏名','APソース丸め','sai_flg','kaisu','組手','本交渉設定週','saikosho_apsource','社数カウント']].copy()
honkosho_settei_company = honkosho_settei_company.loc[honkosho_settei_company["社数カウント"] == 1]
honkosho_settei_company = honkosho_settei_company.rename(columns={"kosho_setteibi":"日付","kohosha_tanto":"担当者","本交渉設定週":"ｶﾚﾝﾀﾞｰ週",'候_略氏名':'sei_plus'}).drop(["社数カウント"], axis=1)
honkosho_settei_company["value"] = honkosho_settei_company.apply(lambda x : 1 if x["kaisu"] == 1 else 0 ,axis = 1)
honkosho_settei_company['type'] = 'settei_com'

#本交渉実施だけのテーブル
honkosho_jisshi = honkosho_data[['id','kosho_jisshibi','kohosha_tanto','候_略氏名','APソース丸め','sai_flg','kaisu','組手','本交渉実施週','saikosho_apsource']].copy()
honkosho_jisshi = honkosho_jisshi.rename(columns={"kosho_jisshibi":"日付","kohosha_tanto":"担当者","本交渉実施週":"ｶﾚﾝﾀﾞｰ週",'候_略氏名':'sei_plus'}).dropna(subset=["日付"])
honkosho_jisshi["value"] = honkosho_jisshi.apply(lambda x : 1 if x["kaisu"] == 1 else 0 ,axis = 1)
honkosho_jisshi['type'] = 'jissi'

honkosho_jisshi_kohosha = honkosho_data[['id','kosho_jisshibi','kohosha_tanto','候_略氏名','APソース丸め','sai_flg','kaisu','組手','本交渉実施週','saikosho_apsource','候補者重複']].copy()
honkosho_jisshi_kohosha = honkosho_jisshi_kohosha.dropna(subset=["kosho_jisshibi"])
honkosho_jisshi_kohosha = honkosho_jisshi_kohosha.loc[honkosho_jisshi_kohosha["候補者重複"] == 1]
honkosho_jisshi_kohosha = honkosho_jisshi_kohosha.rename(columns={"kosho_jisshibi":"日付","kohosha_tanto":"担当者","本交渉実施週":"ｶﾚﾝﾀﾞｰ週",'候_略氏名':'sei_plus'}).drop(["候補者重複"], axis=1)
honkosho_jisshi_kohosha["value"] = honkosho_jisshi_kohosha.apply(lambda x : 1 if x["kaisu"] == 1 else 0 ,axis = 1)
honkosho_jisshi_kohosha['type'] = 'jissi_kohosha'

honkosho_jisshi_company = honkosho_data[['id','kosho_jisshibi','kohosha_tanto','候_略氏名','APソース丸め','sai_flg','kaisu','組手','本交渉実施週','saikosho_apsource','社数カウント']].copy()
honkosho_jisshi_company = honkosho_jisshi_company.dropna(subset=["kosho_jisshibi"])
honkosho_jisshi_company = honkosho_jisshi_company.loc[honkosho_jisshi_company["社数カウント"] == 1]
honkosho_jisshi_company = honkosho_jisshi_company.rename(columns={"kosho_jisshibi":"日付","kohosha_tanto":"担当者","本交渉実施週":"ｶﾚﾝﾀﾞｰ週",'候_略氏名':'sei_plus'}).drop(["社数カウント"], axis=1)
honkosho_jisshi_company["value"] = honkosho_jisshi_company.apply(lambda x : 1 if x["kaisu"] == 1 else 0 ,axis = 1)
honkosho_jisshi_company['type'] = 'jissi_com'


In [ ]:
#企業担当側
#本交渉設定だけのテーブル
honkosho_kigyo_settei = honkosho_data[['id','kosho_setteibi','kigyo_tanto','企_略氏名','APソース丸め','sai_flg','kaisu','組手','本交渉設定週','saikosho_apsource']].copy()
honkosho_kigyo_settei = honkosho_kigyo_settei.rename(columns={"kosho_setteibi":"日付","kigyo_tanto":"担当者","本交渉設定週":"ｶﾚﾝﾀﾞｰ週",'企_略氏名':'sei_plus'})
honkosho_kigyo_settei["value"] = honkosho_kigyo_settei.apply(lambda x : 1 if x["kaisu"] == 1 else 0 ,axis = 1)
honkosho_kigyo_settei['type'] = 'kigyo_settei'

honkosho_kigyo_settei_kohosha = honkosho_data[['id','kosho_setteibi','kigyo_tanto','企_略氏名','APソース丸め','sai_flg','kaisu','組手','本交渉設定週','saikosho_apsource','候補者重複']].copy()
honkosho_kigyo_settei_kohosha = honkosho_kigyo_settei_kohosha.loc[honkosho_kigyo_settei_kohosha["候補者重複"] == 1]
honkosho_kigyo_settei_kohosha = honkosho_kigyo_settei_kohosha.rename(columns={"kosho_setteibi":"日付","kigyo_tanto":"担当者","本交渉設定週":"ｶﾚﾝﾀﾞｰ週",'企_略氏名':'sei_plus'}).drop(["候補者重複"], axis=1)
honkosho_kigyo_settei_kohosha["value"] = honkosho_kigyo_settei_kohosha.apply(lambda x : 1 if x["kaisu"] == 1 else 0 ,axis = 1)
honkosho_kigyo_settei_kohosha['type'] = 'kigyo_settei_kohosha'

honkosho_kigyo_settei_company = honkosho_data[['id','kosho_setteibi','kigyo_tanto','企_略氏名','APソース丸め','sai_flg','kaisu','組手','本交渉設定週','saikosho_apsource','社数カウント']].copy()
honkosho_kigyo_settei_company = honkosho_kigyo_settei_company.loc[honkosho_kigyo_settei_company["社数カウント"] == 1]
honkosho_kigyo_settei_company = honkosho_kigyo_settei_company.rename(columns={"kosho_setteibi":"日付","kigyo_tanto":"担当者","本交渉設定週":"ｶﾚﾝﾀﾞｰ週",'企_略氏名':'sei_plus'}).drop(["社数カウント"], axis=1)
honkosho_kigyo_settei_company["value"] = honkosho_kigyo_settei_company.apply(lambda x : 1 if x["kaisu"] == 1 else 0 ,axis = 1)
honkosho_kigyo_settei_company['type'] = 'kigyo_settei_com'

#本交渉実施だけのテーブル
honkosho_kigyo_jisshi = honkosho_data[['id','kosho_jisshibi','kigyo_tanto','企_略氏名','APソース丸め','sai_flg','kaisu','組手','本交渉実施週','saikosho_apsource']].copy()
honkosho_kigyo_jisshi = honkosho_kigyo_jisshi.rename(columns={"kosho_jisshibi":"日付","kigyo_tanto":"担当者","本交渉実施週":"ｶﾚﾝﾀﾞｰ週",'企_略氏名':'sei_plus'}).dropna(subset=["日付"])
honkosho_kigyo_jisshi["value"] = honkosho_kigyo_jisshi.apply(lambda x : 1 if x["kaisu"] == 1 else 0 ,axis = 1)
honkosho_kigyo_jisshi['type'] = 'kigyo_jissi'

honkosho_kigyo_jisshi_kohosha = honkosho_data[['id','kosho_jisshibi','kigyo_tanto','企_略氏名','APソース丸め','sai_flg','kaisu','組手','本交渉実施週','saikosho_apsource','候補者重複']].copy()
honkosho_kigyo_jisshi_kohosha = honkosho_kigyo_jisshi_kohosha.dropna(subset=["kosho_jisshibi"])
honkosho_kigyo_jisshi_kohosha = honkosho_kigyo_jisshi_kohosha.loc[honkosho_kigyo_jisshi_kohosha["候補者重複"] == 1]
honkosho_kigyo_jisshi_kohosha = honkosho_kigyo_jisshi_kohosha.rename(columns={"kosho_jisshibi":"日付","kigyo_tanto":"担当者","本交渉実施週":"ｶﾚﾝﾀﾞｰ週",'企_略氏名':'sei_plus'}).drop(["候補者重複"], axis=1)
honkosho_kigyo_jisshi_kohosha["value"] = honkosho_kigyo_jisshi_kohosha.apply(lambda x : 1 if x["kaisu"] == 1 else 0 ,axis = 1)
honkosho_kigyo_jisshi_kohosha['type'] = 'kigyo_jissi_kohosha'

honkosho_kigyo_jisshi_company = honkosho_data[['id','kosho_jisshibi','kigyo_tanto','企_略氏名','APソース丸め','sai_flg','kaisu','組手','本交渉実施週','saikosho_apsource','社数カウント']].copy()
honkosho_kigyo_jisshi_company = honkosho_kigyo_jisshi_company.dropna(subset=["kosho_jisshibi"])
honkosho_kigyo_jisshi_company = honkosho_kigyo_jisshi_company.loc[honkosho_kigyo_jisshi_company["社数カウント"] == 1]
honkosho_kigyo_jisshi_company = honkosho_kigyo_jisshi_company.rename(columns={"kosho_jisshibi":"日付","kigyo_tanto":"担当者","本交渉実施週":"ｶﾚﾝﾀﾞｰ週",'企_略氏名':'sei_plus'}).drop(["社数カウント"], axis=1)
honkosho_kigyo_jisshi_company["value"] = honkosho_kigyo_jisshi_company.apply(lambda x : 1 if x["kaisu"] == 1 else 0 ,axis = 1)
honkosho_kigyo_jisshi_company['type'] = 'kigyo_jissi_com'


In [ ]:
#本交渉設定と実施をそれぞれ同じ形で整える
honkosho = pd.concat([honkosho_settei,honkosho_settei_kohosha],ignore_index = True)
honkosho = pd.concat([honkosho,honkosho_settei_company],ignore_index = True)
honkosho = pd.concat([honkosho,honkosho_jisshi],ignore_index = True)
honkosho = pd.concat([honkosho,honkosho_jisshi_kohosha],ignore_index = True)
honkosho = pd.concat([honkosho,honkosho_jisshi_company],ignore_index = True)

honkosho = pd.concat([honkosho,honkosho_kigyo_settei],ignore_index = True)
honkosho = pd.concat([honkosho,honkosho_kigyo_settei_kohosha],ignore_index = True)
honkosho = pd.concat([honkosho,honkosho_kigyo_settei_company],ignore_index = True)
honkosho = pd.concat([honkosho,honkosho_kigyo_jisshi],ignore_index = True)
honkosho = pd.concat([honkosho,honkosho_kigyo_jisshi_kohosha],ignore_index = True)
honkosho = pd.concat([honkosho,honkosho_kigyo_jisshi_company],ignore_index = True)

#Q情報　追加
honkosho = pd.merge(honkosho,Q_master,on = ("日付"),how = "left")
honkosho.head(3)

honkosho.to_excel('honkosho.xlsx',sheet_name='new_sheet_name')


# 紹介受領

In [ ]:
#sql ="""
#select
#a.id,
#a.kohosha_id,
#b.seimei as kohosha_seimei,
#kohosha_sql.name as kohosha_rank,
#b.not_count as kohosha_not_count,
#apsource_sql.name as kohosha_apsource,
#a.shokaimoto_id,
#-- case when a.ap_source = "20" then c.id ELSE NULL END AS `人事部id`,
#-- case when a.ap_source = "20" then c.company_name ELSE NULL END AS `会社名`,
#-- case when a.ap_source = "20" then c.yakushoku ELSE NULL END AS `役職`,
#-- case when a.ap_source = "10" then d.komon_id else null end as "顧問id",
#-- case when a.ap_source = "10" then d.company_name else null end as "紹介元企業名",
#a.juryosha,
#a.juryobi
#from
#_live_rhs__shokaijuryos as a

#left join
#(select id,seimei,ap_source,kohosha_rank,not_count,shokaimoto_id from _live_rhs__kohoshas) as b ON a.kohosha_id = b.id
#left join
#(select group_code,code,name from _live_rhs__sys_consts where group_code = "19") as apsource_sql ON a.ap_source = apsource_sql.code
#left join
#(select group_code,code,name from _live_rhs__sys_consts where group_code = "5200") as kohosha_sql ON b.kohosha_rank = kohosha_sql.code
#-- left join
#-- (select id,company_name,yakushoku from _live_rhs__jinjibus) as c on b.shokaimoto_id = c.id
#-- left join
#-- (select id,komon_id,company_name from _live_rhs__partners) as d on a.shokaimoto_id = d.id
#;
#"""
#shokaijuryos_data = pd.DataFrame(get_data(sql, conn3))

#候補者ランクが空欄の箇所を埋める
#shokaijuryos_data["kohosha_rank"] =shokaijuryos_data["kohosha_rank"].fillna(0)

#候補者APソースが空欄の行を削除
#shokaijuryos_data = shokaijuryos_data.dropna(subset=["kohosha_apsource"])
#
#shokaijuryos_data["juryobi"] = pd.to_datetime(shokaijuryos_data["juryobi"]) #日付データを変換

In [ ]:
sql ="""
select
a.id,
a.kohosha_id,
b.seimei as kohosha_seimei,
kohosha_sql.name as kohosha_rank,
b.not_count as kohosha_not_count,
apsource_sql.name as kohosha_apsource,
a.shokaimoto_id,
-- case when a.ap_source = "20" then c.id ELSE NULL END AS `人事部id`,
-- case when a.ap_source = "20" then c.company_name ELSE NULL END AS `会社名`,
-- case when a.ap_source = "20" then c.yakushoku ELSE NULL END AS `役職`,
-- case when a.ap_source = "10" then d.komon_id else null end as "顧問id",
-- case when a.ap_source = "10" then d.company_name else null end as "紹介元企業名",
a.juryosha,
a.juryobi
from
`temp-380708.live_rhs.shokaijuryos` as a

left join
(select id,seimei,ap_source,kohosha_rank,not_count,shokaimoto_id from `temp-380708.live_rhs.kohoshas`) as b ON a.kohosha_id = b.id
left join
(select group_code,code,name from `temp-380708.live_rhs.sys_consts` where group_code = 19) as apsource_sql ON a.ap_source = apsource_sql.code
left join
(select group_code,code,name from `temp-380708.live_rhs.sys_consts` where group_code = 5200) as kohosha_sql ON b.kohosha_rank = kohosha_sql.code
-- left join
-- (select id,company_name,yakushoku from `temp-380708.live_rhs.jinjibus`) as c on b.shokaimoto_id = c.id
-- left join
-- (select id,komon_id,company_name from `temp-380708.live_rhs.partners`) as d on a.shokaimoto_id = d.id

where a.juryobi >= '2017-04-01'

;
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
shokaijuryos_data = client.query(sql).result().to_dataframe()

#候補者ランクが空欄の箇所を埋める
shokaijuryos_data["kohosha_rank"] =shokaijuryos_data["kohosha_rank"].fillna(0)

#候補者APソースが空欄の行を削除
shokaijuryos_data = shokaijuryos_data.dropna(subset=["kohosha_apsource"])

shokaijuryos_data["juryobi"] = pd.to_datetime(shokaijuryos_data["juryobi"]) #日付データを変換

### 紹介受領データを整える

In [ ]:
#候補者APソース　追加
shokaijuryos_data = shokaijuryos_data.rename(columns={"kohosha_apsource":"ヨミ表選択"}) #カラム名変換
shokaijuryos_data = pd.merge(shokaijuryos_data,master4,on = ("ヨミ表選択"),how = "left") #候補者APソースを紐付け
shokaijuryos_data = shokaijuryos_data.rename(columns={"丸め":"APソース丸め"})
#紹介受領週（カレンダー用）　追加
shokaijuryos_data = shokaijuryos_data.rename(columns={"juryobi":"日付"}) #カラム名変換
shokaijuryos_data = pd.merge(shokaijuryos_data,master1,on = ("日付"),how = "left") #初期交渉設定日関連の日付データを紐付け
shokaijuryos_data = shokaijuryos_data.rename(columns={"週":"紹介受領週","計上Q":"Q","日付":"juryobi","juryosha": "user_id"})

shokaijuryos_data = pd.merge(shokaijuryos_data,syain_data,on = "user_id",how="left") #名前が変換される

In [ ]:
shokaijuryos_data = shokaijuryos_data[['id','juryobi','user_id','sei_plus','APソース丸め','紹介受領週']].copy()
shokaijuryos_data = shokaijuryos_data.rename(columns={"juryobi":"日付","user_id":"担当者","紹介受領週":"ｶﾚﾝﾀﾞｰ週"})

shokaijuryos_data["value"] = 1
shokaijuryos_data['type'] = 'shokaijuryos'
shokaijuryos_data['組手'] = ''
shokaijuryos_data['sai_flg'] = ''
shokaijuryos_data['kaisu'] = 1
shokaijuryos_data['saikosho_apsource'] = shokaijuryos_data['APソース丸め']

#Q情報　追加
shokaijuryos = pd.merge(shokaijuryos_data,Q_master,on = ("日付"),how = "left")
shokaijuryos.head()

shokaijuryos.to_excel('shokaijuryos.xlsx',sheet_name='new_sheet_name')


# 企業AP・企業営業

In [ ]:
#営業AP数のデータを出す
#sql = """
#SELECT
#id,
#kigyo_id,
#appoint_get_syain,
#appoint_visit_syain,
#appoint_get_date,
#visit_times as kaisu,
#company_name,
#appoint_source,
#appoint_visit_plan_date
#FROM
#_live_rhs__sales_appoints;
#"""
#sales_ap_data = pd.DataFrame(get_data(sql, conn3))

#sales_ap_data["appoint_get_date"] = pd.to_datetime(sales_ap_data["appoint_get_date"]) #日付データを変換
#sales_ap_data["appoint_visit_plan_date"] = pd.to_datetime(sales_ap_data["appoint_visit_plan_date"]) #日付データを変換


In [ ]:
#営業AP数のデータを出す
sql = """
SELECT
id,
kigyo_id,
appoint_get_syain,
appoint_visit_syain,
appoint_get_date,
visit_times as kaisu,
company_name,
appoint_source,
appoint_visit_plan_date
FROM
`temp-380708.live_rhs.sales_appoints`

where appoint_get_date >= '2017-04-01'


;
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
sales_ap_data = client.query(sql).result().to_dataframe()

sales_ap_data["appoint_get_date"] = pd.to_datetime(sales_ap_data["appoint_get_date"]) #日付データを変換
sales_ap_data["appoint_visit_plan_date"] = pd.to_datetime(sales_ap_data["appoint_visit_plan_date"]) #日付データを変換


In [ ]:
#企業初期商談設定週（カレンダー用）　追加
sales_ap_data = sales_ap_data.rename(columns={"appoint_get_date":"日付"}) #カラム名変換
sales_ap_data = pd.merge(sales_ap_data,master1,on = ("日付"),how = "left") #対面商談設定日関連の日付データを紐付け
sales_ap_data = sales_ap_data.rename(columns={"週":"対面商談設定週","日付":"appoint_get_date","appoint_get_syain":"user_id","計上Q":"Q"})
sales_ap_data = pd.merge(sales_ap_data,syain_data,on = "user_id",how="left") #名前が変換される

sales_ap_data = sales_ap_data[['id','appoint_get_date','user_id',"sei_plus",'対面商談設定週','kaisu']].copy()
sales_ap_data = sales_ap_data.rename(columns={"appoint_get_date":"日付","user_id":"担当者","対面商談設定週":"ｶﾚﾝﾀﾞｰ週"})

sales_ap_data["kaisu"] =sales_ap_data["kaisu"].fillna(0)
sales_ap_data["value"] = sales_ap_data.apply(lambda x : 1 if x["kaisu"] == 1 else 0 ,axis = 1)
sales_ap_data['type'] = 'eigyou_ap'
sales_ap_data['組手'] = ''
sales_ap_data['sai_flg'] = ''
sales_ap_data['APソース丸め'] = ''

#Q情報　追加
sales_ap = pd.merge(sales_ap_data,Q_master,on = ("日付"),how = "left")


In [ ]:
#営業数のデータを抽出

#sql = """
#SELECT
#id,
#kigyo_id,
#company_name,
#appoint_source,
#appoint_visit_syain,
#fulfills_date,
#visit_times as kaisu
#FROM
#_live_rhs__sales_appoint_fulfills;

#"""

#sales_apful_data = pd.DataFrame(get_data(sql, conn3))
#sales_apful_data["fulfills_date"] = pd.to_datetime(sales_apful_data["fulfills_date"]) #日付データを変換


In [ ]:
#営業数のデータを抽出

sql = """
SELECT
id,
kigyo_id,
company_name,
appoint_source,
appoint_visit_syain,
fulfills_date,
visit_times as kaisu
FROM
`temp-380708.live_rhs.sales_appoint_fulfills`

where fulfills_date >= '2017-04-01'

;

"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
sales_apful_data = client.query(sql).result().to_dataframe()

sales_apful_data["fulfills_date"] = pd.to_datetime(sales_apful_data["fulfills_date"]) #日付データを変換

In [ ]:
#企業営業実施週（カレンダー用）　追加
sales_apful_data = sales_apful_data.rename(columns={"fulfills_date":"日付"}) #カラム名変換
sales_apful_data = pd.merge(sales_apful_data,master1,on = ("日付"),how = "left") #対面商談設定日関連の日付データを紐付け
sales_apful_data = sales_apful_data.rename(columns={"週":"アポ実施週","日付":"fulfills_date","appoint_visit_syain":"user_id"})
sales_apful_data = pd.merge(sales_apful_data,syain_data,on = "user_id",how="left") #名前が変換される

sales_apful_data = sales_apful_data[['id','fulfills_date','user_id',"sei_plus",'アポ実施週','kaisu']].copy()
sales_apful_data = sales_apful_data.rename(columns={"fulfills_date":"日付","user_id":"担当者","アポ実施週":"ｶﾚﾝﾀﾞｰ週"})

sales_apful_data["kaisu"] =sales_apful_data["kaisu"].fillna(0)

sales_apful_data["value"] = sales_apful_data.apply(lambda x : 1 if x["kaisu"] == 1 else 0 ,axis = 1)
sales_apful_data['type'] = 'eigyou'
sales_apful_data['組手'] = ''
sales_apful_data['sai_flg'] = ''
sales_apful_data['APソース丸め'] = ''
sales_apful_data['saikosho_apsource'] = ''

#Q情報　追加
sales_apful = pd.merge(sales_apful_data,Q_master,on = ("日付"),how = "left")

sales_apful.to_excel('sales_apful.xlsx',sheet_name='new_sheet_name')

In [ ]:
df = pd.concat([shokikosho, honkosho])
df = pd.concat([df, shokaijuryos])
df = pd.concat([df, sales_ap])
df = pd.concat([df, sales_apful])

df.head(2)


In [ ]:
df.to_excel('df.xlsx',sheet_name='new_sheet_name')

In [ ]:
#　ロンザン事業部全体数値用 (初接触時のアポソース別)
df_all = df[df["期間対象外"] == "対象"]
df1 = df_all.pivot_table(index=["type","APソース丸め"],columns="月",aggfunc="sum",values="value").fillna(0)
df2 = df_all.pivot_table(index=["type","APソース丸め"],columns="Q",aggfunc="sum",values="value").fillna(0)
df3 = df_all.pivot_table(index=["type","APソース丸め"],columns="Q同営",aggfunc="sum",values="value").fillna(0)
df4 = df_all.pivot_table(index=["type","APソース丸め"],columns="Q同旬",aggfunc="sum",values="value").fillna(0)
concat_all=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_all["type"] = concat_all["type"]+concat_all["APソース丸め"]


#　ロンザン事業部全体数値用 (shokikosho　の　再交渉分だけカウント)
df_sai_shoki = df[(df["sai_flg"] == "再交渉") & (df["期間対象外"] == "対象") & (df["type"].str.contains('shoki'))]
df1 = df_sai_shoki.pivot_table(index=["type","sai_flg"],columns="月",aggfunc="count",values="value").fillna(0)
df2 = df_sai_shoki.pivot_table(index=["type","sai_flg"],columns="Q",aggfunc="count",values="value").fillna(0)
df3 = df_sai_shoki.pivot_table(index=["type","sai_flg"],columns="Q同営",aggfunc="count",values="value").fillna(0)
df4 = df_sai_shoki.pivot_table(index=["type","sai_flg"],columns="Q同旬",aggfunc="count",values="value").fillna(0)
concat_sai_shoki=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_sai_shoki["type"] = concat_sai_shoki["type"]+"_sai"+ concat_sai_shoki["sai_flg"]


#　ロンザン事業部全体数値用 (honkosho　の　再交渉部分だけカウント)
df_sai_honkosho = df[(df["sai_flg"] == "再交渉") & (df["期間対象外"] == "対象")  & (~df["type"].str.contains('shoki'))]
df1 = df_sai_honkosho.pivot_table(index=["type","sai_flg"],columns="月",aggfunc="sum",values="value").fillna(0)
df2 = df_sai_honkosho.pivot_table(index=["type","sai_flg"],columns="Q",aggfunc="sum",values="value").fillna(0)
df3 = df_sai_honkosho.pivot_table(index=["type","sai_flg"],columns="Q同営",aggfunc="sum",values="value").fillna(0)
df4 = df_sai_honkosho.pivot_table(index=["type","sai_flg"],columns="Q同旬",aggfunc="sum",values="value").fillna(0)
concat_sai_honkosho=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_sai_honkosho["type"] = concat_sai_honkosho["type"]+"_sai"+ concat_sai_honkosho["sai_flg"]


# ロンザン事業部全体数値用（組手別）
df_kumite = df[df["期間対象外"] == "対象"]
df1 = df_kumite.pivot_table(index=["type","組手"],columns="月",aggfunc="sum",values="value").fillna(0)
df2 = df_kumite.pivot_table(index=["type","組手"],columns="Q",aggfunc="sum",values="value").fillna(0)
df3 = df_kumite.pivot_table(index=["type","組手"],columns="Q同営",aggfunc="sum",values="value").fillna(0)
df4 = df_kumite.pivot_table(index=["type","組手"],columns="Q同旬",aggfunc="sum",values="value").fillna(0)
concat_kumite=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_kumite["type"] = concat_kumite["type"]+concat_kumite["組手"]

concat_all = pd.merge(concat_all,concat_sai_shoki,how = "outer")
concat_all = pd.merge(concat_all,concat_sai_honkosho,how = "outer")
concat_all = pd.merge(concat_all,concat_kumite,how = "outer")

In [ ]:
#　個人別数値用
df_kojin = df[df["期間対象外"] == "対象"]
df1 = df_kojin.pivot_table(index=["type","sei_plus"],columns="月",aggfunc="sum",values="value").fillna(0)
df2 = df_kojin.pivot_table(index=["type","sei_plus"],columns="Q",aggfunc="sum",values="value").fillna(0)
df3 = df_kojin.pivot_table(index=["type","sei_plus"],columns="Q同営",aggfunc="sum",values="value").fillna(0)
df4 = df_kojin.pivot_table(index=["type","sei_plus"],columns="Q同旬",aggfunc="sum",values="value").fillna(0)
concat_kojin=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_kojin["type"] = concat_kojin["type"]+concat_kojin["sei_plus"]

#　個人別数値用（組手別）
df_kojin_kumite = df[df["期間対象外"] == "対象"]
df1 = df_kojin_kumite.pivot_table(index=["type","組手","sei_plus"],columns="月",aggfunc="sum",values="value").fillna(0)
df2 = df_kojin_kumite.pivot_table(index=["type","組手","sei_plus"],columns="Q",aggfunc="sum",values="value").fillna(0)
df3 = df_kojin_kumite.pivot_table(index=["type","組手","sei_plus"],columns="Q同営",aggfunc="sum",values="value").fillna(0)
df4 = df_kojin_kumite.pivot_table(index=["type","組手","sei_plus"],columns="Q同旬",aggfunc="sum",values="value").fillna(0)
concat_kojin_kumite=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_kojin_kumite["type"] = concat_kojin_kumite["type"]+concat_kojin_kumite["組手"]+concat_kojin_kumite["sei_plus"]

#　個人別数値用（紹介アポ獲得数（人））
df_kojin_shoki_ap_partner = df[(df["APソース丸め"] == "パートナー紹介") & (df["期間対象外"] == "対象") & (df["type"].str.contains('shoki_ap'))]
df1 = df_kojin_shoki_ap_partner.pivot_table(index=["type","sei_plus"],columns="月",aggfunc="sum",values="value").fillna(0)
df2 = df_kojin_shoki_ap_partner.pivot_table(index=["type","sei_plus"],columns="Q",aggfunc="sum",values="value").fillna(0)
df3 = df_kojin_shoki_ap_partner.pivot_table(index=["type","sei_plus"],columns="Q同営",aggfunc="sum",values="value").fillna(0)
df4 = df_kojin_shoki_ap_partner.pivot_table(index=["type","sei_plus"],columns="Q同旬",aggfunc="sum",values="value").fillna(0)
concat_kojin_shoki_ap_partner=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_kojin_shoki_ap_partner["type"] = concat_kojin_shoki_ap_partner["type"] + "_partner" +concat_kojin_shoki_ap_partner["sei_plus"]

#　ロンザン事業部全体数値用 (shokikosho　の　再交渉分だけカウント)
df_kojin_shoki = df[(df["sai_flg"] == "再交渉") & (df["期間対象外"] == "対象") & (df["type"].str.contains('shoki'))]
df1 = df_sai_shoki.pivot_table(index=["type","sai_flg","sei_plus"],columns="月",aggfunc="count",values="value").fillna(0)
df2 = df_sai_shoki.pivot_table(index=["type","sai_flg","sei_plus"],columns="Q",aggfunc="count",values="value").fillna(0)
df3 = df_sai_shoki.pivot_table(index=["type","sai_flg","sei_plus"],columns="Q同営",aggfunc="count",values="value").fillna(0)
df4 = df_sai_shoki.pivot_table(index=["type","sai_flg","sei_plus"],columns="Q同旬",aggfunc="count",values="value").fillna(0)
concat_kojin_shoki=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_kojin_shoki["type"] = concat_kojin_shoki["type"]+"_sai"+concat_kojin_shoki["sei_plus"]

In [ ]:
#　ロンザン事業部全体数値用 (honkosho　の　再交渉部分だけカウント)
df_kojin_sai_honkosho = df[(df["sai_flg"] == "再交渉") & (df["期間対象外"] == "対象")  & (~df["type"].str.contains('shoki'))]

In [ ]:
df1 = df_kojin_sai_honkosho.pivot_table(index=["type","sai_flg","sei_plus"],columns="月",aggfunc="sum",values="value").fillna(0)
df2 = df_kojin_sai_honkosho.pivot_table(index=["type","sai_flg","sei_plus"],columns="Q",aggfunc="sum",values="value").fillna(0)
df3 = df_kojin_sai_honkosho.pivot_table(index=["type","sai_flg","sei_plus"],columns="Q同営",aggfunc="sum",values="value").fillna(0)
df4 = df_kojin_sai_honkosho.pivot_table(index=["type","sai_flg","sei_plus"],columns="Q同旬",aggfunc="sum",values="value").fillna(0)

In [ ]:
concat_kojin_sai_honkosho=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()

In [ ]:
concat_kojin_sai_honkosho["type"] = concat_kojin_sai_honkosho["type"]+"_sai"+ concat_kojin_sai_honkosho["sei_plus"]

In [ ]:
concat_kojin

In [ ]:
concat_kojin = pd.merge(concat_kojin,concat_kojin_kumite,how = "outer")
concat_kojin = pd.merge(concat_kojin,concat_kojin_shoki_ap_partner,how = "outer")
concat_kojin = pd.merge(concat_kojin,concat_kojin_shoki,how = "outer")
concat_kojin = pd.merge(concat_kojin,concat_kojin_sai_honkosho,how = "outer")

In [ ]:
concat_kojin

In [ ]:
#  カレンダー用
#アポソース別（再交渉になった案件は、アポソース別の数値から除く　※初回が人事部、その後再交渉案件の場合は、　人事部のカウント0、再交渉側のアポソースで1）
df_calendar = df[(df["期間対象外"] == "対象") & (df["sai_flg"] != "再交渉")]
calendar = df_calendar.pivot_table(index=["type","APソース丸め","sei_plus","sai_flg"],columns="ｶﾚﾝﾀﾞｰ週",aggfunc="sum",values="value").fillna(0).reset_index()
calendar["type"] = calendar["type"]+calendar["APソース丸め"]+calendar["sei_plus"]

#再交渉用（再交渉になったアポソースのものをカウント　※初回が人事部、その後再交渉案件の場合は、　人事部のカウント0、再交渉側のアポソースで1）
df_calendar_sai = df[(df["sai_flg"] == "再交渉") & (df["期間対象外"] == "対象") & (df["kaisu"] == 1)]
calendar_sai = df_calendar_sai.pivot_table(index=["type","sei_plus","sai_flg"],columns="ｶﾚﾝﾀﾞｰ週",aggfunc="count",values="value").fillna(0).reset_index()
calendar_sai["type"] = calendar_sai["type"]+"_sai"+calendar_sai["sei_plus"]

#再交渉用（再交渉になったアポソースのものをカウント　shokiko部分のみカウント）
df_calendar_sai_shoki = df[(df["sai_flg"] == "再交渉") & (df["期間対象外"] == "対象") & (df["type"].str.contains('shoki'))]
calendar_sai_shoki = df_calendar_sai_shoki.pivot_table(index=["type","sei_plus","sai_flg"],columns="ｶﾚﾝﾀﾞｰ週",aggfunc="count",values="value").fillna(0).reset_index()
calendar_sai_shoki["type"] = calendar_sai_shoki["type"]+"_sai"+calendar_sai_shoki["sei_plus"]


#企業設定・実施用
df_calendar_kigyo = df[df["期間対象外"] == "対象"]
calendar_kigyo = df_calendar_kigyo.pivot_table(index=["type","sei_plus"],columns="ｶﾚﾝﾀﾞｰ週",aggfunc="sum",values="value").fillna(0).reset_index()
calendar_kigyo["type"] = calendar_kigyo["type"]+"_all_1"+calendar_kigyo["sei_plus"]

#企業設定・実施用(2回目をカウント)
df_calendar_kigyo_jissi = df[(df["kaisu"] == 2) & (df["期間対象外"] == "対象") ]
calendar_kigyo_jissi = df_calendar_kigyo_jissi.pivot_table(index=["type","sei_plus"],columns="ｶﾚﾝﾀﾞｰ週",aggfunc="count",values="value").fillna(0).reset_index()
calendar_kigyo_jissi["type"] = calendar_kigyo_jissi["type"]+"_all_2"+calendar_kigyo_jissi["sei_plus"]

calendar = pd.merge(calendar_sai,calendar,how = "outer")
calendar = pd.merge(calendar,calendar_sai_shoki,how = "outer")
calendar = pd.merge(calendar,calendar_kigyo,how = "outer")
calendar = pd.merge(calendar,calendar_kigyo_jissi,how = "outer")

In [ ]:
#with pd.ExcelWriter('index.xlsx', engine = 'xlsxwriter') as writer:
#    concat_all.to_excel(writer,sheet_name='all')
#    concat_kojin.to_excel(writer,sheet_name='kojin')
#    calendar.to_excel(writer,sheet_name='calendar')

In [ ]:
#calendar.to_excel('calendar.xlsx',sheet_name='new_sheet_name')

# 顧客支持pt、受注数 

In [ ]:
col_name = ["案件id\n（RZ）","計上月","期","月","案件\nNo\n（SC）","入力者","計上日","受注日","クライアント正式名称","候補者",
            "売上種別（商品内容）","差分\n（該当場合のみ）","紹介引当/PM引当\n特殊ポイント\n（該当場合のみ）","基準年収","報酬率",
            "完保\n成約\n割振比","グループ企画料","サービス\n引当\n係数","キャンセル\n引当\n係数","営業売上合計","所属課",
            "氏名","割合","受注額","査定用\n売上","顧客支持ポイント","引き継ぎP","担当","担当\n押印","備考①","備考②","備考③","シニアスカウト\n（ヨミ表と一致）",
            "内定数フラグ","企業アポソース","特殊フラグ","提示/前年度","基準年収.1","報酬率.1","d","d.1","アポソース"]

path1 = r"\\172.16.0.232\CoffeeCrazy\経営ソリューション事業部\□シニアスカウト事業部□\01 全体進捗\02 行動カレンダー\pythonデータ"

#過去のポイントデータ（こちらは積み上げ式にしているのでExcelファイルからもってくる　※RPAとBTのクロスセル分は除いた状態にて積み上げ済）
yomi_old_betu = pd.read_excel(path1 + "\yojitu\RZポイント表過去データ（22-3Qまで）最新.xlsx",usecols=[0,1,2,3,4])
yomi_old_betu = yomi_old_betu.rename(columns={"両手案件フラグ":"旧両手","アポソース丸め":"旧アポ",
                                                "候補者アポソース":"旧候補者アポ","候補者id":"旧候id"})

yomi_old = pd.read_excel(path1 + "\yojitu\RZポイント表過去データ（22-3Qまで）最新.xlsx",usecols  =[5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47])[col_name]
yomi_old = yomi_old.rename(columns={"営業売上\n（営業ポイント）.1":"営業売上\n（営業ポイント）"})
#yomi_old.to_excel('22-3Qデータ.xlsx')


In [ ]:
yomi_old.head(1)

In [ ]:
#今Qのポイントデータは最新のヨミ表からとってくる
yomi_nowQ = pd.read_excel(r"\\172.16.0.232\CoffeeCrazy\管理グループ\管理部\ポイント割り振り表\【入力用】ポイント表\26期\26-3\ロンザン統合版ポイント表\【26-3Q】入力用顧客支持ポイント表（ロンザン）.xlsm",header=20,usecols = (range(0, 43)))
yomi_nowQ = yomi_nowQ[:-1]

#RPA事業部のクロスセルは除く(顧問名が「塩澤昌紘」はRPA事業部のクロスセル案件)
yomi_nowQ = yomi_nowQ[~yomi_nowQ['候補者'].isin(['塩澤昌紘'])]
yomi_nowQ = yomi_nowQ[~yomi_nowQ['売上種別（商品内容）'].isin(['RPAコンサル'])]
yomi_nowQ = yomi_nowQ[~yomi_nowQ['売上種別（商品内容）'].isin(['ビジネスタンク'])]
yomi_nowQ.to_excel('yomi_nowQ.xlsx')

yomi_nowQ.head(2)

In [ ]:
#過去計上済のヨミ表情報と、今Qのヨミ表とをくっつける
yomi = pd.concat([yomi_old,yomi_nowQ],axis=0,ignore_index=True)

In [ ]:
#ポイントデータを結合する
yomi_data =yomi.copy()
yomi_data.index = yomi_data.index + 1
yomi_data = yomi_data.reset_index()
yomi_data = yomi_data.set_index("index")

#上からナンバーを割り振る
yomi_data = yomi_data.reset_index()
yomi_data = yomi_data.rename(columns={"index":"発番"})
#yomi_data.to_excel('総合ポイント.xlsx')

yomi_data.head(1)

In [ ]:
#honkosho_settei_data = honkosho_data[['id','kosho_setteibi','kohosha_id','APソース丸め','組手','sai_flg','anken_id']].copy()
#a = honkosho_settei_data[honkosho_settei_data['id'] == 16964]
#a

案件情報（本交渉情報）からアポソースや組手情報などを紐付ける

In [ ]:
#設定元データと最新ポイントファイルを結合 (アポソース等を紐付けるため)
honkosho_settei_data = honkosho_data[['id','kosho_setteibi','kohosha_id','APソース丸め','組手','sai_flg','anken_id']].copy()
honkosho_settei_data = honkosho_settei_data.rename(columns={"anken_id":"案件id\n（RZ）"})
honkosho_settei_data= honkosho_settei_data.drop_duplicates(subset=["案件id\n（RZ）"],keep='first')
    
yomi_data["案件id\n（RZ）"]=pd.to_numeric(yomi_data["案件id\n（RZ）"],errors='coerce')
yomi_data["案件id\n（RZ）"]=yomi_data["案件id\n（RZ）"].fillna(0.0).astype(int)
yomi_data["案件id\n（RZ）"]=yomi_data["案件id\n（RZ）"].astype(int)

yomi_data = pd.merge(yomi_data,honkosho_settei_data,on = "案件id\n（RZ）",how="left")

#ヨミ表データと過去のヨミ表データを紐付ける（候補者アポソース取得のため）
yomi_data = pd.merge(yomi_data,yomi_old_betu,on = "発番",how="left")

#企業担当フラグ（この成約の企業担当にフラグを付ける）
yomi_data["企業担当フラグ"] = yomi_data.apply(lambda x : 1 if x["担当"] in ("面談担当","面談担当①","面談担当・候補者担当") else 0 ,axis = 1)

#候補者IDが紐づかない受注については、過去使った候補者IDを紐付ける
yomi_data["kohosha_id"] = yomi_data["kohosha_id"].fillna(0.0)
yomi_data["候補者id"] = yomi_data.apply(lambda x : x["旧候id"] if x["kohosha_id"] == 0 else x["kohosha_id"] ,axis = 1)



In [ ]:
#本交渉IDが紐づかない受注については、過去使ったアポソースを紐付ける）
yomi_data["APソース丸め"] = yomi_data["APソース丸め"].fillna(0.0)
yomi_data["候補者APソース2"] = yomi_data.apply(lambda x : x["旧アポ"] if x["APソース丸め"] == 0 else x["APソース丸め"] ,axis = 1)

#ヨミ表のアポソース丸めデータと紐付ける
yomi_data = yomi_data.rename(columns={"候補者APソース2":"ヨミ表選択"})
yomi_data["ヨミ表選択"]=yomi_data["ヨミ表選択"].fillna(0.0)
yomi_data = pd.merge(yomi_data,master4,on = ("ヨミ表選択"),how = "left")
yomi_data["丸め"] = yomi_data["丸め"].fillna(0.0)
yomi_data["候補者アポソース丸め"] = yomi_data.apply(lambda x : "■クロスセル" if x["丸め"] == 0 else x["丸め"] ,axis = 1)
yomi_data = yomi_data.drop(["丸め"], axis=1)

In [ ]:
#過去ポイントファイルの計上日と担当者をマスタデータと紐付けるため、名前を変換
yomi_data = yomi_data.rename(columns={"計上日":"日付","月":"計上日月"})

#カレンダー用の週データを紐付ける
yomi_data["日付"] = pd.to_datetime(yomi_data["日付"])
yomi_data = pd.merge(yomi_data,master1,on = ("日付"),how = "left")
yomi_data = yomi_data.drop(["月"], axis=1).rename(columns={"週":"計上週"})

#Qマスタのデータと紐付ける（同営業日・同旬月日など）
Q_master["日付"] = pd.to_datetime(Q_master["日付"])
yomi_data = pd.merge(yomi_data,Q_master,on = ("日付"),how = "left")

In [ ]:
#当時のQの職種を抽出するためにQと担当者名を紐付ける
mas = master2[['Q','sei_plus','所属フラグ','ロンザン所属フラグ']].copy()
mas["Q所属フラグ"] = mas["Q"] + mas["sei_plus"]
mas= mas.drop_duplicates(subset=["Q所属フラグ"],keep='first').drop(["Q"], axis=1)

yomi_data["Q所属フラグ"] = yomi_data["計上Q"] + yomi_data["氏名"]
yomi_data["Q所属フラグ"]=yomi_data["Q所属フラグ"].fillna(0.0)

yomi_data = pd.merge(yomi_data,mas,on = ("Q所属フラグ"),how = "left")
yomi_data= yomi_data.rename(columns={"所属フラグ":"担当所属フラグ","月":"計上月末","日付マスタ":"計上日"})
yomi_data = yomi_data.drop(['sei_plus'],axis=1)

In [ ]:
yomi_data.head(1)

In [ ]:
#過去計上Qの掛け率などをこちらで設定
yomi_data["計上Q"]=yomi_data["計上Q"].fillna(0.0)
yomi_data = pd.merge(yomi_data,master5,on = ("計上Q"),how = "left")

yomi_data["顧客支持ポイント"]=pd.to_numeric(yomi_data["顧客支持ポイント"],errors='coerce')
yomi_data["顧客支持ポイント"]=yomi_data["顧客支持ポイント"].fillna(0.0).astype(float)
yomi_data["ポイント丸め"] = yomi_data.apply(lambda x : 1 if x["掛け率"] == "0" else x["掛け率"] * x["顧客支持ポイント"] ,axis = 1)

#yomi_data["ポイント"] = yomi_data["掛け率"] * yomi_data["顧客支持ポイント"] #"営業売上\n（営業ポイント）"（丸め前のデータ）
#yomi_data["ポイント丸め"] = yomi_data.apply(lambda x : 1 if x["掛け率"] == "0" else x["ポイント"] ,axis = 1)

yomi_data = yomi_data.rename(columns={"案件id\n（RZ）":"案件id（RZ）","シニアスカウト\n（ヨミ表と一致）":"シニアスカウト"})

In [ ]:
#両手企業、片手企業のフラグを作成
#候補者担当がRZ所属（退職者含む）、企業担当RZ所属フロントミドル→両手
#候補者担当がRZ所属（退職者含む）以外、企業担当RZ所属フロントミドル→片手

rt1 = yomi_data[(yomi_data["企業担当フラグ"] == 1) & (yomi_data["所属課"] == 'ロンザン') ]
rt1 = rt1.loc[:,["案件id（RZ）","企業担当フラグ","所属課"]]
rt1["案件id（RZ）"] = rt1.apply(lambda x : "-" if x["案件id（RZ）"] == 0 else x["案件id（RZ）"] ,axis = 1)
rt1["企業担当ロンザンか"] = 1.0
rt1 = rt1.loc[:,["案件id（RZ）",'企業担当ロンザンか']]
rt1["案件id（RZ）"] = rt1["案件id（RZ）"].astype(str)
rt1 = rt1.drop_duplicates(subset=["案件id（RZ）"],keep='first')

rt2 = yomi_data[(yomi_data["内定数フラグ"] == 1) & (yomi_data["所属課"] == 'ロンザン') ]
rt2 = rt2.loc[:,["案件id（RZ）","内定数フラグ","所属課"]]
rt2["案件id（RZ）"] = rt2.apply(lambda x : "-" if x["案件id（RZ）"] == 0 else x["案件id（RZ）"] ,axis = 1)
rt2["候補者担当ロンザンか"] = 1.0
rt2 = rt2.loc[:,["案件id（RZ）",'候補者担当ロンザンか']]
rt2["案件id（RZ）"] = rt2["案件id（RZ）"].astype(str)
rt2 = rt2.drop_duplicates(subset=["案件id（RZ）"],keep='first')

rt = pd.merge(rt1,rt2,on = ("案件id（RZ）"),how = "left")
del rt1,rt2

rt["企業フラグ"] = rt["企業担当ロンザンか"] * rt["候補者担当ロンザンか"]
rt["企業フラグ"] = rt["企業フラグ"].apply(lambda x : "両手" if x == 1.0 else "片手")
rt = rt.drop(['企業担当ロンザンか','候補者担当ロンザンか'],axis=1)


In [ ]:
#両手企業フラグのデータをポイントデータに紐付け
yomi_data["案件id（RZ）"] = yomi_data["案件id（RZ）"].astype(str)
yomi_data = pd.merge(yomi_data,rt,on = ("案件id（RZ）"),how = "left")

#企業フラグ　→　ヨミ表上の候補者担当と企業担当情報から両手かどうか判断しているフラグ
yomi_data["両手案件フラグ"] = yomi_data.apply(lambda x : "両手" if x["企業フラグ"] == "両手" else "片手" ,axis = 1)

#旧両手　→　本交渉に組手情報がない時の案件、かつ、過去に片手か両手か判断した案件
yomi_data["旧両手"] = yomi_data["旧両手"].fillna(0)
yomi_data["両手案件フラグ"] = yomi_data.apply(lambda x : x["旧両手"] if x["旧両手"] != 0 else x["両手案件フラグ"] ,axis = 1)

#組手　→　本交渉データを元にした組手（組手情報がない場合は、案件データの企業担当＆候補者担当がロンザンメンバーかどうかをみて判断）
yomi_data["組手"] = yomi_data["組手"].fillna(0)
yomi_data["両手案件フラグ"] = yomi_data.apply(lambda x : x["組手"] if x["組手"] != 0 else x["両手案件フラグ"] ,axis = 1)

In [ ]:
#yomi_data_ = yomi_data.loc[:,["候補者担当フラグ","企業担当フラグ","氏名","担当所属フラグ","ロンザン所属フラグ",
#                        "候補者アポソース丸め","計上Q","計上月","計上月末","計上週","ポイント丸め","sai_flg","両手案件フラグ",
#                       "ヨミ表選択","期間対象外","Q同営","Q同旬",
#                        "kohosha_id","案件id（RZ）","期","案件\nNo\n（SC）",
#                        "入力者","日付","受注日","クライアント正式名称","候補者","売上種別（商品内容）","基準年収",
#                        "報酬率","新ポイント用","営業売上合計","所属課","氏名","割合","受注額","営業売上\n（営業ポイント）",
#                        "引き継ぎP","担当","備考①","備考②","備考③","内定数フラグ"]]

rzp = yomi_data.rename(columns={"案件\nNo\n（SC）":"案件No（SC）"})

rzp["計上月末"] = pd.to_datetime(rzp["計上月末"])
rzp = rzp.astype({'候補者アポソース丸め': str, '計上Q': str})

各数値算出（顧客支持ポイント、成約数）　ロンザンALL用

In [ ]:
#APソースが再交渉以外のAPソースの内訳をポイント（エージェントのみ）抽出
#パートナー、人事部、顧問、SMAP、上場企業、転機、その他のポイント（エージェントのみ）抽出
rzp2 = rzp[(rzp["期間対象外"] == "対象") & (rzp["所属課"] == "ロンザン")]
rzp2['type'] = "point_ag"
rzp2.loc[:,"type"] = rzp2["type"].astype(str)
df1 = rzp2.pivot_table(index=["type","候補者アポソース丸め"],columns="計上月末",aggfunc="sum",values="ポイント丸め").fillna(0)
df2 = rzp2.pivot_table(index=["type","候補者アポソース丸め"],columns="計上Q",aggfunc="sum",values="ポイント丸め").fillna(0)
df3 = rzp2.pivot_table(index=["type","候補者アポソース丸め"],columns="Q同営",aggfunc="sum",values="ポイント丸め").fillna(0)
df4 = rzp2.pivot_table(index=["type","候補者アポソース丸め"],columns="Q同旬",aggfunc="sum",values="ポイント丸め").fillna(0)
concat_pt_all=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_pt_all["type"] = concat_pt_all["type"]+concat_pt_all["候補者アポソース丸め"]

#APソースが再交渉の内訳をポイント（エージェントのみ）抽出
rzp5 = rzp[(rzp["sai_flg"] == "再交渉") & (rzp["期間対象外"] == "対象")  & (rzp["所属課"] == "ロンザン")]
rzp5["type"] = "point_ag_sai"
rzp5.loc[:,"type"] = rzp5["type"].astype(str)
df1 = rzp5.pivot_table(index=["type"],columns="計上月末",aggfunc="sum",values="ポイント丸め").fillna(0)
df2 = rzp5.pivot_table(index=["type"],columns="計上Q",aggfunc="sum",values="ポイント丸め").fillna(0)
df3 = rzp5.pivot_table(index=["type"],columns="Q同営",aggfunc="sum",values="ポイント丸め").fillna(0)
df4 = rzp5.pivot_table(index=["type"],columns="Q同旬",aggfunc="sum",values="ポイント丸め").fillna(0)
concat_pt_sai=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_pt_sai["type"] = concat_pt_sai["type"]+"再交渉"

#アポソース別成約数
rzp6 = rzp[(rzp["期間対象外"] == "対象") & (rzp["内定数フラグ"] == 1)]
rzp6['type'] = "naitei"
rzp6.loc[:,"type"] = rzp6["type"].astype(str)
df1 = rzp6.pivot_table(index=["type","候補者アポソース丸め"],columns="計上月末",aggfunc="count",values="内定数フラグ").fillna(0)
df2 = rzp6.pivot_table(index=["type","候補者アポソース丸め"],columns="計上Q",aggfunc="count",values="内定数フラグ").fillna(0)
df3 = rzp6.pivot_table(index=["type","候補者アポソース丸め"],columns="Q同営",aggfunc="count",values="内定数フラグ").fillna(0)
df4 = rzp6.pivot_table(index=["type","候補者アポソース丸め"],columns="Q同旬",aggfunc="count",values="内定数フラグ").fillna(0)
concat_naitei=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_naitei["type"] = concat_naitei["type"]+concat_naitei["候補者アポソース丸め"]

#片手両手別の成約数
rzp6['type'] = rzp6.apply(lambda x : "ryoute" if x["両手案件フラグ"] == "両手" else "katate" ,axis = 1)
rzp6.loc[:,"type"] = rzp6["type"].astype(str)
df1 = rzp6.pivot_table(index=["type","両手案件フラグ"],columns="計上月末",aggfunc="count",values="内定数フラグ").fillna(0)
df2 = rzp6.pivot_table(index=["type","両手案件フラグ"],columns="計上Q",aggfunc="count",values="内定数フラグ").fillna(0)
df3 = rzp6.pivot_table(index=["type","両手案件フラグ"],columns="Q同営",aggfunc="count",values="内定数フラグ").fillna(0)
df4 = rzp6.pivot_table(index=["type","両手案件フラグ"],columns="Q同旬",aggfunc="count",values="内定数フラグ").fillna(0)
concat_kumite=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_kumite["type"] = concat_kumite["type"] + "_seiyaku総計"

#再交渉の成約数
rzp7 = rzp[(rzp["sai_flg"] == "再交渉") & (rzp["期間対象外"] == "対象") & (rzp["内定数フラグ"] == 1)]
rzp7['type'] = "naitei_sai"
rzp7.loc[:,"type"] = rzp7["type"].astype(str)
df1 = rzp7.pivot_table(index=["type","sai_flg"],columns="計上月末",aggfunc="count",values="内定数フラグ").fillna(0)
df2 = rzp7.pivot_table(index=["type","sai_flg"],columns="計上Q",aggfunc="count",values="内定数フラグ").fillna(0)
df3 = rzp7.pivot_table(index=["type","sai_flg"],columns="Q同営",aggfunc="count",values="内定数フラグ").fillna(0)
df4 = rzp7.pivot_table(index=["type","sai_flg"],columns="Q同旬",aggfunc="count",values="内定数フラグ").fillna(0)
concat_naitei_sai=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_naitei_sai["type"] = concat_naitei_sai["type"]+concat_naitei_sai["sai_flg"]

#平均基準年収・報酬率抽出
rzp10 = rzp[(rzp["期間対象外"] == "対象") & (rzp["内定数フラグ"] == 1)]

#基準年収
rzp10['type'] = "income"
rzp10["基準年収"] = rzp10["基準年収"].fillna(0.0)

#アポソース別平均基準年収抽出（候補者アポソース丸め＝本交渉テーブルのアポソースを、ヨミ表に加工したもの。再交渉案件の場合も、アポソースは元のアポソース(案件に紐づくアポソース)のまま）
rzp10.loc[:,"type"] = rzp10["type"].astype(str)
df1 = rzp10.pivot_table(index=["type","候補者アポソース丸め"],columns="計上月末",aggfunc="mean",values="基準年収").fillna(0)
df2 = rzp10.pivot_table(index=["type","候補者アポソース丸め"],columns="計上Q",aggfunc="mean",values="基準年収").fillna(0)
df3 = rzp10.pivot_table(index=["type","候補者アポソース丸め"],columns="Q同営",aggfunc="mean",values="基準年収").fillna(0)
df4 = rzp10.pivot_table(index=["type","候補者アポソース丸め"],columns="Q同旬",aggfunc="mean",values="基準年収").fillna(0)
concat_income_apsource=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_income_apsource["type"] = concat_income_apsource["type"]+concat_income_apsource["候補者アポソース丸め"]

#ロンザン全体の平均基準年収
df1 = rzp10.pivot_table(index=["type"],columns="計上月末",aggfunc="mean",values="基準年収").fillna(0)
df2 = rzp10.pivot_table(index=["type"],columns="計上Q",aggfunc="mean",values="基準年収").fillna(0)
df3 = rzp10.pivot_table(index=["type"],columns="Q同営",aggfunc="mean",values="基準年収").fillna(0)
df4 = rzp10.pivot_table(index=["type"],columns="Q同旬",aggfunc="mean",values="基準年収").fillna(0)
concat_income_all=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_income_all["type"] = concat_income_all["type"] + "総計"

#片手両手別の平均基準年収
rzp10['type'] = rzp10.apply(lambda x : "ryoute" if x["両手案件フラグ"] == "両手" else "katate" ,axis = 1)
df1 = rzp10.pivot_table(index=["type","両手案件フラグ"],columns="計上月末",aggfunc="mean",values="基準年収").fillna(0)
df2 = rzp10.pivot_table(index=["type","両手案件フラグ"],columns="計上Q",aggfunc="mean",values="基準年収").fillna(0)
df3 = rzp10.pivot_table(index=["type","両手案件フラグ"],columns="Q同営",aggfunc="mean",values="基準年収").fillna(0)
df4 = rzp10.pivot_table(index=["type","両手案件フラグ"],columns="Q同旬",aggfunc="mean",values="基準年収").fillna(0)
concat_income_kumite=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_income_kumite["type"] = concat_income_kumite["type"] + "_income総計"

#報酬率
rzp10['type'] = "hoshu_ritsu"
rzp10["報酬率"] = rzp10["報酬率"].fillna(0.0)

#アポソース別報酬率抽出（候補者アポソース丸め＝本交渉テーブルのアポソースを、ヨミ表に加工したもの。再交渉案件の場合も、アポソースは元のアポソース(案件に紐づくアポソース)のまま）
rzp10.loc[:,"type"] = rzp10["type"].astype(str)
df1 = rzp10.pivot_table(index=["type","候補者アポソース丸め"],columns="計上月末",aggfunc="mean",values="報酬率").fillna(0)
df2 = rzp10.pivot_table(index=["type","候補者アポソース丸め"],columns="計上Q",aggfunc="mean",values="報酬率").fillna(0)
df3 = rzp10.pivot_table(index=["type","候補者アポソース丸め"],columns="Q同営",aggfunc="mean",values="報酬率").fillna(0)
df4 = rzp10.pivot_table(index=["type","候補者アポソース丸め"],columns="Q同旬",aggfunc="mean",values="報酬率").fillna(0)
concat_hoshu_ritsu_apsource=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_hoshu_ritsu_apsource["type"] = concat_hoshu_ritsu_apsource["type"]+concat_hoshu_ritsu_apsource["候補者アポソース丸め"]

#ロンザン全体の報酬率
df1 = rzp10.pivot_table(index=["type"],columns="計上月末",aggfunc="mean",values="報酬率").fillna(0)
df2 = rzp10.pivot_table(index=["type"],columns="計上Q",aggfunc="mean",values="報酬率").fillna(0)
df3 = rzp10.pivot_table(index=["type"],columns="Q同営",aggfunc="mean",values="報酬率").fillna(0)
df4 = rzp10.pivot_table(index=["type"],columns="Q同旬",aggfunc="mean",values="報酬率").fillna(0)
concat_hoshu_ritsu_all=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_hoshu_ritsu_all["type"] = concat_hoshu_ritsu_all["type"] + "総計"

#片手両手別の報酬率
rzp10['type'] = rzp10.apply(lambda x : "ryoute" if x["両手案件フラグ"] == "両手" else "katate" ,axis = 1)
df1 = rzp10.pivot_table(index=["type","両手案件フラグ"],columns="計上月末",aggfunc="mean",values="報酬率").fillna(0)
df2 = rzp10.pivot_table(index=["type","両手案件フラグ"],columns="計上Q",aggfunc="mean",values="報酬率").fillna(0)
df3 = rzp10.pivot_table(index=["type","両手案件フラグ"],columns="Q同営",aggfunc="mean",values="報酬率").fillna(0)
df4 = rzp10.pivot_table(index=["type","両手案件フラグ"],columns="Q同旬",aggfunc="mean",values="報酬率").fillna(0)
concat_hoshu_ritsu_kumite=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_hoshu_ritsu_kumite["type"] = concat_hoshu_ritsu_kumite["type"] + "_hoshu_ritsu総計"

個人別数値算出用

In [ ]:
#個人別顧客支持pt（クロスセルも含む※ただし、BT・RPAは除く）
rzpk2 = rzp[(rzp["期間対象外"] == "対象") & (rzp["所属課"] == "ロンザン")]
rzpk2['type'] = "point_ag"
rzpk2.loc[:,"type"] = rzpk2["type"].astype(str)
df1 = rzpk2.pivot_table(index=["type","氏名"],columns="計上月末",aggfunc="sum",values="ポイント丸め").fillna(0)
df2 = rzpk2.pivot_table(index=["type","氏名"],columns="計上Q",aggfunc="sum",values="ポイント丸め").fillna(0)
df3 = rzpk2.pivot_table(index=["type","氏名"],columns="Q同営",aggfunc="sum",values="ポイント丸め").fillna(0)
df4 = rzpk2.pivot_table(index=["type","氏名"],columns="Q同旬",aggfunc="sum",values="ポイント丸め").fillna(0)
concat_pt_kojin=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_pt_kojin["type"] = concat_pt_kojin["type"]+concat_pt_kojin["氏名"].astype(str)

#クロスセルポイント抽出
rzpk4 = rzp[ (rzp["期間対象外"] == "対象") & (rzp["候補者アポソース丸め"] == "■クロスセル")  & (rzp["所属課"] == "ロンザン")]
rzpk4['type'] = "point_cross_all"
rzpk4.loc[:,"type"] = rzpk4["type"].astype(str)
df1 = rzpk4.pivot_table(index=["type","氏名"],columns="計上月末",aggfunc="sum",values="ポイント丸め").fillna(0)
df2 = rzpk4.pivot_table(index=["type","氏名"],columns="計上Q",aggfunc="sum",values="ポイント丸め").fillna(0)
df3 = rzpk4.pivot_table(index=["type","氏名"],columns="Q同営",aggfunc="sum",values="ポイント丸め").fillna(0)
df4 = rzpk4.pivot_table(index=["type","氏名"],columns="Q同旬",aggfunc="sum",values="ポイント丸め").fillna(0)
concat_pt_cross_kojin=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_pt_cross_kojin["type"] = concat_pt_cross_kojin["type"]+concat_pt_cross_kojin["氏名"].astype(str)

#候補者担当別成約数
rzpk6 = rzp[(rzp["期間対象外"] == "対象") & (rzp["内定数フラグ"] == 1)]

rzpk6['type'] = "naitei"
rzpk6.loc[:,"type"] = rzpk6["type"].astype(str)
df1 = rzpk6.pivot_table(index=["type","氏名"],columns="計上月末",aggfunc="count",values="内定数フラグ").fillna(0)
df2 = rzpk6.pivot_table(index=["type","氏名"],columns="計上Q",aggfunc="count",values="内定数フラグ").fillna(0)
df3 = rzpk6.pivot_table(index=["type","氏名"],columns="Q同営",aggfunc="count",values="内定数フラグ").fillna(0)
df4 = rzpk6.pivot_table(index=["type","氏名"],columns="Q同旬",aggfunc="count",values="内定数フラグ").fillna(0)
concat_naitei_kojin=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_naitei_kojin["type"] = concat_naitei_kojin["type"]+concat_naitei_kojin["氏名"].astype(str)

#片手両手別の成約数
rzpk6['type'] = rzpk6.apply(lambda x : "ryoute" if x["両手案件フラグ"] == "両手" else "katate" ,axis = 1)
rzpk6.loc[:,"type"] = rzpk6["type"].astype(str)
df1 = rzpk6.pivot_table(index=["type","両手案件フラグ","氏名"],columns="計上月末",aggfunc="count",values="内定数フラグ").fillna(0)
df2 = rzpk6.pivot_table(index=["type","両手案件フラグ","氏名"],columns="計上Q",aggfunc="count",values="内定数フラグ").fillna(0)
df3 = rzpk6.pivot_table(index=["type","両手案件フラグ","氏名"],columns="Q同営",aggfunc="count",values="内定数フラグ").fillna(0)
df4 = rzpk6.pivot_table(index=["type","両手案件フラグ","氏名"],columns="Q同旬",aggfunc="count",values="内定数フラグ").fillna(0)
concat_kumite_kojin=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_kumite_kojin["type"] = concat_kumite_kojin["type"] + "_seiyaku" + concat_kumite_kojin["氏名"].astype(str)

#候補者担当別平均基準年収
rzpk6['type'] = "income"
df1 = rzpk6.pivot_table(index=["type","氏名"],columns="計上月末",aggfunc="mean",values="基準年収").fillna(0)
df2 = rzpk6.pivot_table(index=["type","氏名"],columns="計上Q",aggfunc="mean",values="基準年収").fillna(0)
df3 = rzpk6.pivot_table(index=["type","氏名"],columns="Q同営",aggfunc="mean",values="基準年収").fillna(0)
df4 = rzpk6.pivot_table(index=["type","氏名"],columns="Q同旬",aggfunc="mean",values="基準年収").fillna(0)
concat_income_kojin=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_income_kojin["type"] = concat_income_kojin["type"] + concat_income_kojin["氏名"].astype(str)

#片手両手別の報酬率
rzpk6['type'] = "hoshu_ritsu"
df1 = rzpk6.pivot_table(index=["type","氏名"],columns="計上月末",aggfunc="mean",values="報酬率").fillna(0)
df2 = rzpk6.pivot_table(index=["type","氏名"],columns="計上Q",aggfunc="mean",values="報酬率").fillna(0)
df3 = rzpk6.pivot_table(index=["type","氏名"],columns="Q同営",aggfunc="mean",values="報酬率").fillna(0)
df4 = rzpk6.pivot_table(index=["type","氏名"],columns="Q同旬",aggfunc="mean",values="報酬率").fillna(0)
concat_hoshu_ritsu_kojin=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_hoshu_ritsu_kojin["type"] = concat_hoshu_ritsu_kojin["type"] + concat_hoshu_ritsu_kojin["氏名"].astype(str)


#片手両手別の平均基準年収
rzpk6['type'] = rzpk6.apply(lambda x : "ryoute" if x["両手案件フラグ"] == "両手" else "katate" ,axis = 1)
df1 = rzpk6.pivot_table(index=["type","両手案件フラグ","氏名"],columns="計上月末",aggfunc="mean",values="基準年収").fillna(0)
df2 = rzpk6.pivot_table(index=["type","両手案件フラグ","氏名"],columns="計上Q",aggfunc="mean",values="基準年収").fillna(0)
df3 = rzpk6.pivot_table(index=["type","両手案件フラグ","氏名"],columns="Q同営",aggfunc="mean",values="基準年収").fillna(0)
df4 = rzpk6.pivot_table(index=["type","両手案件フラグ","氏名"],columns="Q同旬",aggfunc="mean",values="基準年収").fillna(0)
concat_income_kumite_kojin=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_income_kumite_kojin["type"] = concat_income_kumite_kojin["type"] + "_income" + concat_income_kumite_kojin["氏名"].astype(str)

#片手両手別の報酬率
rzpk6['type'] = rzpk6.apply(lambda x : "ryoute" if x["両手案件フラグ"] == "両手" else "katate" ,axis = 1)
df1 = rzpk6.pivot_table(index=["type","両手案件フラグ","氏名"],columns="計上月末",aggfunc="mean",values="報酬率").fillna(0)
df2 = rzpk6.pivot_table(index=["type","両手案件フラグ","氏名"],columns="計上Q",aggfunc="mean",values="報酬率").fillna(0)
df3 = rzpk6.pivot_table(index=["type","両手案件フラグ","氏名"],columns="Q同営",aggfunc="mean",values="報酬率").fillna(0)
df4 = rzpk6.pivot_table(index=["type","両手案件フラグ","氏名"],columns="Q同旬",aggfunc="mean",values="報酬率").fillna(0)
concat_hoshu_ritsu_kumite_kojin=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_hoshu_ritsu_kumite_kojin["type"] = concat_hoshu_ritsu_kumite_kojin["type"] + "_hoshu_ritsu" + concat_hoshu_ritsu_kumite_kojin["氏名"].astype(str)

#企業担当別成約数
rzpk7 = rzp[(rzp["期間対象外"] == "対象") & (rzp["企業担当フラグ"] == 1)]
rzpk7['type'] = "naitei_kigyo"
rzpk7.loc[:,"type"] = rzpk7["type"].astype(str)
df1 = rzpk7.pivot_table(index=["type","氏名"],columns="計上月末",aggfunc="count",values="企業担当フラグ").fillna(0)
df2 = rzpk7.pivot_table(index=["type","氏名"],columns="計上Q",aggfunc="count",values="企業担当フラグ").fillna(0)
df3 = rzpk7.pivot_table(index=["type","氏名"],columns="Q同営",aggfunc="count",values="企業担当フラグ").fillna(0)
df4 = rzpk7.pivot_table(index=["type","氏名"],columns="Q同旬",aggfunc="count",values="企業担当フラグ").fillna(0)
concat_naitei_kigyo_kojin=  pd.concat([df1,df2,df3,df4],axis=1).reset_index()
concat_naitei_kigyo_kojin["type"] = concat_naitei_kigyo_kojin["type"]+concat_naitei_kigyo_kojin["氏名"].astype(str)



カレンダー用

In [ ]:
#候補者担当×アポソース別成約数（再交渉になった案件は、元のアポソース側ではなく、再交渉側でカウントするため、再交渉案件を除く）
rzpc5 = rzp[(rzp["期間対象外"] == "対象") & (rzp["内定数フラグ"] == 1) & (rzp["sai_flg"] != 1)]
rzpc5['type'] = "naitei"
rzpc5.loc[:,"type"] = rzpc5["type"].astype(str)
calendar_naitei_cal_kojin = rzpc5.pivot_table(index=["type","氏名","候補者アポソース丸め"],columns="計上週",aggfunc="count",values="内定数フラグ").fillna(0).reset_index()
calendar_naitei_cal_kojin["type"] = calendar_naitei_cal_kojin["type"]+calendar_naitei_cal_kojin["候補者アポソース丸め"]+calendar_naitei_cal_kojin["氏名"].astype(str)

#候補者担当×再交渉成約数
rzpc6 = rzp[(rzp["期間対象外"] == "対象") & (rzp["内定数フラグ"] == 1) & (rzp["sai_flg"] == 1)]
rzpc6['type'] = "naitei再交渉"
rzpc6.loc[:,"type"] = rzpc6["type"].astype(str)
calendar_naitei_cal_sai = rzpc6.pivot_table(index=["type","氏名"],columns="計上週",aggfunc="count",values="内定数フラグ").fillna(0).reset_index()
calendar_naitei_cal_sai["type"] = calendar_naitei_cal_sai["type"]+calendar_naitei_cal_sai["氏名"].astype(str)


#候補者担当×再交渉成約数
rzpc7 = rzp[(rzp["期間対象外"] == "対象") & (rzp["企業担当フラグ"] == 1)]
rzpc7['type'] = "naitei_kigyo"
rzpc7.loc[:,"type"] = rzpc7["type"].astype(str)
calendar_naitei_cal_kigyo = rzpc7.pivot_table(index=["type","氏名"],columns="計上週",aggfunc="count",values="企業担当フラグ").fillna(0).reset_index()
calendar_naitei_cal_kigyo["type"] = calendar_naitei_cal_kigyo["type"]+calendar_naitei_cal_kigyo["氏名"].astype(str)


In [ ]:
#ロンザンALL用
concat_all = pd.merge(concat_all,concat_pt_all,how = "outer")
concat_all = pd.merge(concat_all,concat_pt_sai,how = "outer")
concat_all = pd.merge(concat_all,concat_naitei,how = "outer")
concat_all = pd.merge(concat_all,concat_naitei_sai,how = "outer")
concat_all = pd.merge(concat_all,concat_kumite,how = "outer")
concat_all = pd.merge(concat_all,concat_income_apsource,how = "outer")
concat_all = pd.merge(concat_all,concat_income_all,how = "outer")
concat_all = pd.merge(concat_all,concat_income_kumite,how = "outer")
concat_all = pd.merge(concat_all,concat_hoshu_ritsu_apsource,how = "outer")
concat_all = pd.merge(concat_all,concat_hoshu_ritsu_all,how = "outer")
concat_all = pd.merge(concat_all,concat_hoshu_ritsu_kumite,how = "outer")

In [ ]:
#個人別用
concat_kojin = pd.merge(concat_kojin,concat_pt_kojin,how = "outer")
concat_kojin = pd.merge(concat_kojin,concat_pt_cross_kojin,how = "outer")
concat_kojin = pd.merge(concat_kojin,concat_naitei_kojin,how = "outer")
concat_kojin = pd.merge(concat_kojin,concat_naitei_kigyo_kojin,how = "outer")
concat_kojin = pd.merge(concat_kojin,concat_kumite_kojin,how = "outer")
concat_kojin = pd.merge(concat_kojin,concat_income_kumite_kojin,how = "outer")
concat_kojin = pd.merge(concat_kojin,concat_hoshu_ritsu_kumite_kojin,how = "outer")
concat_kojin = pd.merge(concat_kojin,concat_income_kojin,how = "outer")
concat_kojin = pd.merge(concat_kojin,concat_hoshu_ritsu_kojin,how = "outer")
concat_kojin = pd.merge(concat_kojin,concat_hoshu_ritsu_kojin,how = "outer")



In [ ]:
concat_kojin

In [ ]:
#カレンダー用
calendar = pd.merge(calendar,calendar_naitei_cal_kojin,how = "outer")
calendar = pd.merge(calendar,calendar_naitei_cal_sai,how = "outer")
calendar = pd.merge(calendar,calendar_naitei_cal_kigyo,how = "outer")

In [ ]:
yomi_data.to_excel('yomi_data.xlsx',sheet_name='new_sheet_name')


In [ ]:
with pd.ExcelWriter('index.xlsx', engine = 'xlsxwriter') as writer:
    concat_all.to_excel(writer,sheet_name='all')
    concat_kojin.to_excel(writer,sheet_name='kojin')
    calendar.to_excel(writer,sheet_name='calendar')

In [ ]:
#calendar.to_excel('calendar.xlsx',sheet_name='calendar')
#concat_kojin.to_excel('concat_kojin.xlsx',sheet_name='kojin')

In [ ]:
#calendar.to_excel('calendar.xlsx',sheet_name='new_sheet_name')